<a href="https://colab.research.google.com/github/Imesh-Isuranga/Statistical-Learning-e20154/blob/main/Bayesian_Inference/Bayesian_Inference_Assignment_Answers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Q. Bayesian Estimation of a User Ability Parameter from Item Responses

An online learning platform presents a user with a sequence of $n$ multiple-choice questions **one at a time**. Each question is either answered correctly or incorrectly, allowing the platform to update its estimate of the user's ability dynamically after every response.

Let $Y_i$ denote the user's response to the $i$-th item encountered:

$$Y_i=
\begin{cases}
1, & \text{if the user answers item } i \text{ correctly},\\
0, & \text{if the user answers item } i \text{ incorrectly}.
\end{cases}$$

The platform assumes that the probability of a correct response is governed by a two-parameter logistic (2PL) item response model. Specifically, conditional on the user's latent ability parameter $\Theta=\theta$, the response probability for item $i$ is:

$$P(Y_i=1\mid \Theta=\theta)=p_i(\theta)=\frac{1}{1+e^{-a_i(\theta-b_i)}},$$

where $a_i>0$ is the known discrimination parameter, and $b_i$ is the known difficulty parameter of item $i$.

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running vector of observed responses** up to the current step $k$ (where $1 \le k \le n$).

Before observing any responses, the platform initializes the user's latent ability estimate with a standard normal prior distribution:

$$f_{\Theta}^{(0)}(\theta) = \frac{1}{\sqrt{2\pi}} \exp\left(-\frac{\theta^2}{2}\right) \quad \text{implying} \quad \Theta \sim \mathscr{N}(0,1).$$

As the user progresses, the posterior distribution at step $k-1$ serves as the prior distribution for step $k$.

---

### Tasks

1. **Visualizing the Mechanics:** Plot $P(Y_i=1\mid \Theta=\theta)$ vs $\theta$ using Plotly for two distinct values of $a_i$, where one of those $a_i$ values is paired with three different difficulty values of $b_i$. Interpret how moving $b_i$ shifts the curve horizontally.
2. **Sequential Likelihood Contribution:** Write down the likelihood contribution $L(y_k \mid \theta)$ of a *single* new response $y_k$ at step $k$, given the latent ability $\theta$. Then, write down the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.
3. **Mathematical Formulation of the Running Update:** Write down the recursive relationship for the posterior density at step $k$, denoted $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$, up to a proportionality constant, using the prior state $f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$ and the new observation $y_k$.
4. **Dynamic Shifting:** Explain how a correct answer ($y_k = 1$) to a highly difficult item (large $b_k$) mathematically shifts the peak of the running posterior density distribution relative to the previous step.
5. **Tracking Certainty and Sharpness:** Explain how the discrimination parameter $a_k$ of the current item alters the variance (or "sharpness") of the distribution during a running update. What happens when $a_k$ is very large versus very small?
6. **Numerical Implementation of a Running Grid:** Describe a algorithmic approach to numerically approximate and maintain this running posterior density function on a fixed grid of $\theta$-values. Explicitly state how you would perform the sequential normalization step computationally after an item is answered.


7. **Evaluating Convergence over the Timeline:** Suppose the user's true, hidden latent ability is $\theta_{\text{true}} = 0.75$. Write a Python script that extends your previous grid simulation to track the performance of the running estimators over a sequence of $n = 20$ items.
* **Simulate Responses:** Dynamically generate the user's responses $y_k \in \{0, 1\}$ at each step by comparing a random draw from a Uniform distribution $U(0,1)$ against the true response probability $p_k(\theta_{\text{true}})$. Give each item a random difficulty $b_k \sim \mathscr{N}(0, 1)$ and a random discrimination $a_k \sim \text{Uniform}(0.5, 2.0)$.
* **Track Estimators:** At each step $k$, calculate and store the running Posterior Mean ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$) and the running Maximum A Posteriori ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$) estimate.
* **Visualize:** Use Plotly to create a single line chart showing the progression of both estimators from step $0$ to $20$. Add a static horizontal reference line at $y = 0.75$ representing $\theta_{\text{true}}$.
* **Analysis:** Briefly explain how the distance between your estimators and $\theta_{\text{true}}$ changes as $k$ increases, and interpret what this implies about the platform's confidence in its measurement.


## Sample Answer

### The 2PL Item Response Probability Model

Assuming that the item responses are conditionally independent given $\Theta=\theta$, the likelihood is

$$
P(Y=y\mid \Theta=\theta)=\prod_{i=1}^n
\left[
p_i(\theta)
\right]^{y_i}
\left[
1-p_i(\theta)
\right]^{1-y_i},
$$

where

$$
p_i(\theta)
=\frac{1}{1+e^{-a_i(\theta-b_i)}}.
$$

In [21]:
import numpy as np
import plotly.graph_objects as go

# Define the 2PL Item Response Function
def p_i(theta, a, b):
    return 1 / (1 + np.exp(-a * (theta - b)))

# Generate a range of latent ability values (theta)
theta_vals = np.linspace(-6, 6, 300)

# Define configurations to plot
# Two distinct a_i values (0.5 and 1.5)
# For a_i = 1.5, we include 3 different b_i values (-2, 0, 2)
curves = [
    {"a": 0.5, "b": 0, "line_style": "dash"},
    {"a": 1.5, "b": -2, "line_style": "solid"},
    {"a": 1.5, "b": 0, "line_style": "solid"},
    {"a": 1.5, "b": 2, "line_style": "solid"},
]

# Create the Plotly figure
fig = go.Figure()

for curve in curves:
    a = curve["a"]
    b = curve["b"]
    style = curve["line_style"]

    # Calculate probabilities
    p_vals = p_i(theta_vals, a, b)

    # Add trace to the plot
    fig.add_trace(go.Scatter(
        x=theta_vals,
        y=p_vals,
        mode='lines',
        name=f"a = {a}, b = {b}",
        line=dict(dash=style, width=2.5)
    ))

# Customize layout
fig.update_layout(
    title={
        'text': "Two-Parameter Logistic (2PL) Item Response Curves",
        'y':0.9,
        'x':0.5,
        'xanchor': 'center',
        'yanchor': 'top'
    },
    xaxis_title="Latent Ability (θ)",
    yaxis_title="Probability of Correct Response P(Y_i = 1 | θ)",
    xaxis=dict(range=[-6, 6], gridcolor='rgba(0,0,0,0.1)'),
    yaxis=dict(range=[0, 1.05], gridcolor='rgba(0,0,0,0.1)'),
    template="plotly_white",
    legend=dict(
        yanchor="top",
        y=0.95,
        xanchor="left",
        x=0.05,
        bgcolor="rgba(255,255,255,0.8)"
    )
)

# Display the interactive plot
fig.show()

### Posterior Distribution

At step $k$ (for $k = 1, \dots, n$), the current item response is $y_k$, and its likelihood contribution is:

$$L(y_k \mid \theta) = p_k(\theta)^{y_k} (1 - p_k(\theta))^{1 - y_k}$$

The sequential posterior density of $\Theta$ given the running response history vector $\mathbf{y}^{(k)}$ is updated recursively as follows:

$$f_{\Theta\mid\mathbf{Y}^{(k)}}(\theta\mid\mathbf{y}^{(k)}) = \frac{ \left[ p_k(\theta)^{y_k} (1 - p_k(\theta))^{1 - y_k} \right] f_{\Theta\mid\mathbf{Y}^{(k-1)}}(\theta\mid\mathbf{y}^{(k-1)}) }{ \int_{\mathbb R} \left[ p_k(s)^{y_k} (1 - p_k(s))^{1 - y_k} \right] f_{\Theta\mid\mathbf{Y}^{(k-1)}}(s\mid\mathbf{y}^{(k-1)})\,ds }$$

**Definition of Terms:**

* $f_{\Theta\mid\mathbf{Y}^{(k-1)}}(\theta\mid\mathbf{y}^{(k-1)})$ is the **prior density** for the current step (which is the posterior inherited from the previous step).
* For the very first item ($k=1$), the base prior is the initialized distribution:

$$f_{\Theta\mid\mathbf{Y}^{(0)}}(\theta\mid\mathbf{y}^{(0)}) = f_\Theta^{(0)}(\theta) = \frac{1}{\sqrt{2\pi}}\exp\left(-\frac{\theta^2}{2}\right)$$


* The denominator is the **local normalizing constant**, integrating out $\theta$ across the current likelihood-prior product to ensure the area under the new step-$k$ density curve integrates to 1.

### Baye's Estimate and the MAP Estimate

Under a sequential framework, point estimates like the **Posterior Mean** (Bayes estimate under squared-error loss) or the **Maximum A Posteriori (MAP)** estimate are updated dynamically at each step $k$ using the running posterior density $f_{\Theta\mid\mathbf{Y}^{(k)}}(\theta\mid\mathbf{y}^{(k)})$.

1. Running Posterior Mean (Bayes Estimate)
The Bayesian estimate of the user's latent ability at step $k$, minimizing the expected squared-error loss, is the expected value of the current posterior distribution:
$$\widehat{\theta}_{\mathrm{Bayes}}^{(k)} = \mathbb E[\Theta \mid \mathbf{Y}^{(k)}=\mathbf{y}^{(k)}] = \int_{\mathbb R} \theta \, f_{\Theta\mid\mathbf{Y}^{(k)}}(\theta\mid\mathbf{y}^{(k)})\,d\theta$$

2. Running Maximum A Posteriori (MAP) Estimate
Alternatively, the most likely ability parameter at step $k$ corresponds to the mode (peak) of the current posterior density curve:
$$\widehat{\theta}_{\mathrm{MAP}}^{(k)} \in \arg\max_{\theta\in\mathbb R} f_{\Theta\mid\mathbf{Y}^{(k)}}(\theta\mid\mathbf{y}^{(k)})$$

**Computation Note**

In practice, as the user answers more items, the computational grid tracks the updated array values for $f_{\Theta\mid\mathbf{Y}^{(k)}}(\theta\mid\mathbf{y}^{(k)})$.

* $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ is updated via a running vector dot-product: `np.trapz(theta * current_posterior, theta)`.
* $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$ is extracted simply via the index of the maximum value: `theta[np.argmax(current_posterior)]`.

## Task 1 Interpretation: Difficulty and Discrimination


The 2PL probability is

$$
p_i(\theta)=\frac{1}{1+\exp[-a_i(\theta-b_i)]}.
$$

At $\theta=b_i$,

$$
p_i(b_i)=\frac{1}{1+\exp(0)}=\frac{1}{2}.
$$

Thus, $b_i$ is the ability level at which a user has a $50\%$ chance of answering item $i$ correctly. Increasing $b_i$ shifts the curve to the right, while decreasing $b_i$ shifts it to the left. If $a_i$ is unchanged, changing $b_i$ changes the location but not the basic shape of the curve.

The derivative of the item response probability is

$$
\frac{\partial p_i(\theta)}{\partial\theta}
=a_i p_i(\theta)[1-p_i(\theta)].
$$

At $\theta=b_i$, its slope is

$$
\left.\frac{\partial p_i(\theta)}{\partial\theta}\right|_{\theta=b_i}
=\frac{a_i}{4}.
$$

Therefore, a larger $a_i$ produces a steeper item response curve.

## Task 2: Sequential and Joint Likelihoods


For one response $y_k\in\{0,1\}$, the likelihood is

$$
L(y_k\mid\theta)
=p_k(\theta)^{y_k}[1-p_k(\theta)]^{1-y_k}.
$$

For a correct response, $L(1\mid\theta)=p_k(\theta)$, whereas for an incorrect response, $L(0\mid\theta)=1-p_k(\theta)$.

Assuming conditional independence given $\Theta=\theta$, the joint likelihood of the running history is

$$
L(\mathbf{y}^{(k)}\mid\theta)
=\prod_{i=1}^{k}p_i(\theta)^{y_i}[1-p_i(\theta)]^{1-y_i}.
$$

Its log-likelihood is

$$
\ell_k(\theta)=\sum_{i=1}^{k}\left\{y_i\log p_i(\theta)+(1-y_i)\log[1-p_i(\theta)]\right\}.
$$


## Task 3: Recursive Posterior Update


The posterior inherited from step $k-1$ becomes the prior at step $k$. Hence,

$$
f_{\Theta\mid\mathbf{Y}^{(k)}}(\theta\mid\mathbf{y}^{(k)})
\propto
p_k(\theta)^{y_k}[1-p_k(\theta)]^{1-y_k}
f_{\Theta\mid\mathbf{Y}^{(k-1)}}(\theta\mid\mathbf{y}^{(k-1)}).
$$

The fully normalized update is

$$
f_{\Theta\mid\mathbf{Y}^{(k)}}(\theta\mid\mathbf{y}^{(k)})
=\frac{p_k(\theta)^{y_k}[1-p_k(\theta)]^{1-y_k}f_{k-1}(\theta)}
{\displaystyle\int_{-\infty}^{\infty}p_k(s)^{y_k}[1-p_k(s)]^{1-y_k}f_{k-1}(s)\,ds}.
$$

After all $k$ responses,

$$
f_k(\theta)\propto f_0(\theta)
\prod_{i=1}^{k}p_i(\theta)^{y_i}[1-p_i(\theta)]^{1-y_i}.
$$

The normal prior is not conjugate to the 2PL likelihood, so this posterior normally requires numerical approximation.




## Task 4: Correct Response to a Difficult Item

When $y_k=1$, the update is

$$
f_k(\theta)\propto p_k(\theta)f_{k-1}(\theta).
$$

For a difficult item with a large $b_k$, $p_k(\theta)$ is small at low ability values and larger at high ability values. Multiplication by $p_k(\theta)$ therefore suppresses the lower-ability part of the previous posterior more strongly and shifts probability mass to the right.

This can be shown from

$$
\log f_k(\theta)=\log f_{k-1}(\theta)+\log p_k(\theta)+C
$$

and

$$
\frac{d}{d\theta}\log p_k(\theta)=a_k[1-p_k(\theta)]>0.
$$

At the previous mode $\widehat\theta_{\mathrm{MAP}}^{(k-1)}$, the derivative of $\log f_{k-1}$ is zero. After the correct response, the derivative at that same point is positive:

$$
\left.\frac{d}{d\theta}\log f_k(\theta)\right|_{\widehat\theta_{\mathrm{MAP}}^{(k-1)}}
=a_k\left[1-p_k\left(\widehat\theta_{\mathrm{MAP}}^{(k-1)}\right)\right]>0.
$$

Thus, the new posterior is increasing at the previous mode, and its new peak generally occurs at a higher value of $\theta$. A correct response is especially influential when the item would have been unlikely to be answered correctly under the previous ability estimate.

## Task 5: Discrimination, Information, and Sharpness


The information supplied by item $k$ at ability $\theta$ is

$$
\mathcal{I}_k(\theta)=a_k^2p_k(\theta)[1-p_k(\theta)].
$$

It is maximized at $\theta=b_k$, where $p_k(\theta)=1/2$:

$$
\mathcal{I}_k(b_k)=\frac{a_k^2}{4}.
$$

When $a_k$ is large, the curve is steep. If $b_k$ lies near the plausible ability region, the response supplies considerable information, the posterior becomes sharper, and its variance generally decreases substantially.

When $a_k$ is small, the likelihood is relatively flat over the ability range. It changes the previous posterior only slightly, so the updated posterior remains broader.

A large $a_k$ is not informative everywhere. If the posterior lies far from $b_k$, then $p_k(\theta)$ is near $0$ or $1$ and $p_k(\theta)[1-p_k(\theta)]$ is small.


## Task 6: Fixed-Grid Approximation and Normalization


Choose a grid $\theta_1,\ldots,\theta_G$, for example over $[-6,6]$, and evaluate the initial normal prior on this grid. After response $y_k$ is observed, calculate

$$
p_k(\theta_g)=\frac{1}{1+\exp[-a_k(\theta_g-b_k)]}
$$

and

$$
L_{k,g}=p_k(\theta_g)^{y_k}[1-p_k(\theta_g)]^{1-y_k}.
$$

Form the unnormalized grid posterior

$$
\widetilde f_k(\theta_g)=L_{k,g}f_{k-1}(\theta_g).
$$

Approximate its normalizing constant with the trapezoidal rule:

$$
Z_k\approx\operatorname{Trapz}\left(\widetilde f_k(\theta_g),\theta_g\right).
$$

Then normalize every grid value:

$$
f_k(\theta_g)=\frac{\widetilde f_k(\theta_g)}{Z_k}.
$$

The grid estimators are

$$
\widehat\theta_{\mathrm{Bayes}}^{(k)}
\approx\operatorname{Trapz}\left(\theta_gf_k(\theta_g),\theta_g\right)
$$

and

$$
\widehat\theta_{\mathrm{MAP}}^{(k)}
\approx\theta_{\operatorname{argmax}_g f_k(\theta_g)}.
$$


### Numerical Implementation

In [20]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

# =====================================================================
# PART 1: SEQUENTIAL BAYESIAN UPDATE (MANUAL 4-ITEM SIMULATION)
# =====================================================================

# 1. Define a fine grid of latent ability values (theta)
theta = np.linspace(-5, 5, 500)

# 2. Initialize the prior distribution: Standard Normal N(0, 1)
prior = stats.norm.pdf(theta, 0, 1)

# 3. Define the Item Response Function (2PL Model)
def p_i(theta, a, b):
    return 1 / (1 + np.exp(-a * (theta - b)))

# 4. Define the manual simulated sequence of item encounters
running_items = [
    {"a": 1.0, "b": -1.5, "y": 1},  # Step 1: Got an easy item correct
    {"a": 1.5, "b": 0.5,  "y": 1},  # Step 2: Got a medium-hard item correct
    {"a": 1.2, "b": 1.5,  "y": 0},  # Step 3: Got a very hard item incorrect
    {"a": 2.0, "b": 0.2,  "y": 1}   # Step 4: Got a highly discriminative item correct
]

# Create the first Plotly figure
fig1 = go.Figure()

# Add the initial Prior distribution trace
fig1.add_trace(go.Scatter(
    x=theta,
    y=prior,
    mode='lines',
    name='Initial Prior: N(0,1)',
    line=dict(dash='dash', width=2.5, color='gray')
))

# Run the sequential update loop
current_posterior = prior.copy()

for idx, item in enumerate(running_items):
    a = item["a"]
    b = item["b"]
    y = item["y"]

    # Calculate item response probability across the grid
    prob = p_i(theta, a, b)

    # Calculate the likelihood contribution of this response
    likelihood = (prob ** y) * ((1 - prob) ** (1 - y))

    # Posterior proportional to Prior * Likelihood
    current_posterior = current_posterior * likelihood

    # Numerically normalize the density curve using the trapezoidal rule
    integral = np.trapezoid(current_posterior, theta)
    current_posterior /= integral

    # Format trace labels
    result_text = "Correct" if y == 1 else "Incorrect"
    trace_name = f"Step {idx+1}: After Item {idx+1} ({result_text}, a={a}, b={b})"

    # Add the current posterior step to the plot
    fig1.add_trace(go.Scatter(
        x=theta,
        y=current_posterior,
        mode='lines',
        name=trace_name,
        line=dict(width=2)
    ))

# Customize layout for Figure 1
fig1.update_layout(
    title={
        'text': "Sequential Bayesian Update of User Ability (θ)",
        'y': 0.95, 'x': 0.5, 'xanchor': 'center', 'yanchor': 'top'
    },
    xaxis_title="Latent Ability Parameter (θ)",
    yaxis_title="Probability Density f(θ | y)",
    template="plotly_white",
    hovermode="x unified",
    legend=dict(
        yanchor="top", y=0.95, xanchor="left", x=0.02,
        bgcolor="rgba(255,255,255,0.7)"
    )
)

# Display the first plot
fig1.show()


# =====================================================================
# PART 2: PERFORMANCE TRACKING & CONVERGENCE TIMELINE (20 ITEMS)
# =====================================================================

# Set random seed for reproducibility of the random items/responses
np.random.seed(42)

# 1. Setup Simulation Parameters
theta_true = 0.75
n_items = 20
theta_grid = np.linspace(-5, 5, 1000)  # Finer grid for tracking accuracy

# 2. Generate Random Item Characteristics
a_params = np.random.uniform(0.5, 2.0, size=n_items)
b_params = np.random.normal(0, 1, size=n_items)

# 3. Initialize Tracking Arrays (Step 0 = Initial Prior State)
running_bayes = [0.0]
running_map = [0.0]
steps = list(range(n_items + 1))

# Initialize prior density distribution: N(0, 1)
current_posterior_sim = stats.norm.pdf(theta_grid, 0, 1)

# 4. Run the 20-item Sequential Simulation loop
for k in range(n_items):
    a_k = a_params[k]
    b_k = b_params[k]

    # Compute true response probability at true theta
    prob_true = p_i(theta_true, a_k, b_k)

    # Simulate stochastically generated user response y_k
    y_k = 1 if np.random.uniform(0, 1) < prob_true else 0

    # Calculate likelihood curve across grid
    prob_grid = p_i(theta_grid, a_k, b_k)
    likelihood = (prob_grid ** y_k) * ((1 - prob_grid) ** (1 - y_k))

    # Update and normalize
    current_posterior_sim = current_posterior_sim * likelihood
    integral_sim = np.trapezoid(current_posterior_sim, theta_grid)
    current_posterior_sim /= integral_sim

    # Compute point estimates
    theta_bayes_k = np.trapezoid(theta_grid * current_posterior_sim, theta_grid)
    theta_map_k = theta_grid[np.argmax(current_posterior_sim)]

    # Store estimates
    running_bayes.append(theta_bayes_k)
    running_map.append(theta_map_k)

# 5. Create the second Plotly figure
fig2 = go.Figure()

# Add True Ability Reference Horizontal Line
fig2.add_hline(
    y=theta_true,
    line_dash="dash",
    line_color="red",
    line_width=2,
    annotation_text=f"True Ability (θ = {theta_true})",
    annotation_position="bottom right"
)

# Add Posterior Mean Trace
fig2.add_trace(go.Scatter(
    x=steps, y=running_bayes,
    mode='lines+markers',
    name='Posterior Mean (θ̂_Bayes)',
    line=dict(color='blue', width=2.5),
    marker=dict(size=6)
))

# Add MAP Trace
fig2.add_trace(go.Scatter(
    x=steps, y=running_map,
    mode='lines+markers',
    name='MAP Estimate (θ̂_MAP)',
    line=dict(color='green', width=2),
    marker=dict(size=6, symbol='square')
))

# Customize Layout for Figure 2
fig2.update_layout(
    title={
        'text': "Convergence of Latent Ability Estimators (θ) Over Time",
        'y': 0.93, 'x': 0.5, 'xanchor': 'center', 'yanchor': 'top'
    },
    xaxis_title="Sequence / Item Position (k)",
    yaxis_title="Estimated Ability (θ̂)",
    xaxis=dict(tickmode='linear', tick0=0, dtick=2),
    yaxis=dict(range=[-1, 2]),
    template="plotly_white",
    hovermode="x unified",
    legend=dict(yanchor="top", y=0.15, xanchor="left", x=0.02)
)

# Display the second plot
fig2.show()

## Task 7: Convergence Analysis


At step $0$, the standard normal prior gives

$$
\widehat\theta_{\mathrm{Bayes}}^{(0)}
=\widehat\theta_{\mathrm{MAP}}^{(0)}=0,
$$

so the initial distance from $\theta_{\mathrm{true}}=0.75$ is $0.75$.

As $k$ increases, the response likelihoods accumulate and the posterior is increasingly determined by the user's response pattern. The posterior mean and MAP estimate should therefore generally move toward $0.75$. Their absolute errors are

$$
e_{\mathrm{Bayes}}^{(k)}
=\left|\widehat\theta_{\mathrm{Bayes}}^{(k)}-0.75\right|
$$

and

$$
e_{\mathrm{MAP}}^{(k)}
=\left|\widehat\theta_{\mathrm{MAP}}^{(k)}-0.75\right|.
$$

The errors need not decrease after every item. Responses are random, and an unexpected correct or incorrect answer can temporarily move an estimate away from the true ability. In addition, randomly generated items differ in their information because their $a_k$ and $b_k$ values differ.

The posterior mean is usually smoother because it averages across the entire posterior distribution. The MAP estimate uses only the location of the maximum and may change more abruptly.

As informative responses accumulate, the posterior generally becomes more concentrated and its variance decreases. This narrowing means that the platform is becoming more confident in its measurement. However, 20 items form a relatively small sample, so the estimates are not guaranteed to equal $0.75$ exactly at the final step.


# Q. Bayesian Tracking of Click-Through Rates (CTR) via Conjugate Beta-Binomial Updates

An e-commerce platform wants to optimize its recommendation engine by dynamically estimating the click-through rate (CTR) of a newly launched advertisement. Since user traffic arrives continuously, the platform updates its belief about the advertisement's performance **one impression at a time** rather than waiting for large batch updates.

Let $\Theta = \theta$ represent the true, hidden conversion rate (probability of a click) of the advertisement, where $\theta \in [0, 1]$.

Let $Y_k$ denote a single user's interaction with the advertisement at time step $k$:

$$Y_k =
\begin{cases}
1, & \text{if the user clicks the advertisement}, \\
0, & \text{if the user does not click the advertisement}.
\end{cases}$$

The platform assumes that conditional on the true conversion rate $\Theta = \theta$, each user interaction is an independent Bernoulli trial:

$$P(Y_k = 1 \mid \Theta = \theta) = \theta$$

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running vector of observed user interactions** up to the current impression step $k$ (where $1 \le k \le n$).

Before observing any data, the platform assigns a flexible **Beta distribution** as the initial prior over the unknown parameter $\Theta$:

$$f_{\Theta}^{(0)}(\theta) = \frac{1}{\mathrm{B}(\alpha_0, \beta_0)} \theta^{\alpha_0 - 1} (1 - \theta)^{\beta_0 - 1} \quad \text{implying} \quad \Theta \sim \text{Beta}(\alpha_0, \beta_0)$$

where $\mathrm{B}(\cdot, \cdot)$ is the Beta function acting as the normalizing constant. Under a sequential framework, the posterior distribution at step $k-1$ serves directly as the prior distribution for step $k$.

---

**Tasks**

**1. Structural Probability and Properties**
Plot the probability density function (PDF) of a $\text{Beta}(\alpha, \beta)$ distribution using Plotly for three distinct parameter pairs:

* Uninformative state: $(\alpha=1, \beta=1)$
* Right-skewed state: $(\alpha=2, \beta=8)$
* Left-skewed state: $(\alpha=8, \beta=2)$

Interpret how changing the balance between $\alpha$ and $\beta$ shifts the center of mass of the density function over the domain $[0, 1]$.

**2. Sequential Likelihood and Joint History**

Write down the mathematical likelihood contribution $L(y_k \mid \theta)$ of a *single* isolated response $y_k$ at step $k$, given the click probability $\theta$. Following this, express the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.

**3. Closed-Form Analytical Updates (Conjugacy)**

Using Bayes' Theorem, derive the recursive algebraic relationship for the posterior density at step $k$, denoted as $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$. Prove analytically that the posterior remains in the Beta family (**Beta-Binomial Conjugacy**) by explicitly writing down the closed-form update parameters $\alpha_k$ and $\beta_k$ as simple arithmetic updates of $\alpha_{k-1}$, $\beta_{k-1}$, and $y_k$. Also compute the **Posterior Mean** of the latent parameter $\Theta$ at time step $k$ (i.e. $\mathbb{E}[\Theta \mid \mathbf{Y}^{(k)}=\mathbf{y}^{(k)}]$).


**4. Dynamic Shifting Mechanics**

Explain how an observed click ($y_k = 1$) vs. a non-click ($y_k = 0$) shifts the peak of the running density distribution mathematically. Contrast this analytical framework against non-conjugate setups (such as the 2PL IRT model) where numerical grid integration is strictly required.

**5. Running Point Estimators**

State the exact closed-form equations used to evaluate the following point estimates at step $k$ directly from the updated shape parameters $\alpha_k$ and $\beta_k$:

* **Running Posterior Mean** ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$)
* **Running Maximum A Posteriori** ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$)

**6. Performance Tracking and Convergence Analysis**

Suppose the advertisement's true, hidden click-through rate is $\theta_{\text{true}} = 0.35$. Write a Python script to track the performance of your closed-form sequential estimators over a timeline of $n = 100$ impressions:

* **Initialize State:** Set the base prior parameters to $\alpha_0 = 1, \beta_0 = 1$ (representing uniform initial uncertainty).
* **Simulate Responses:** Dynamically generate user responses $y_k \in \{0, 1\}$ at each step by comparing a random draw from a Uniform distribution $U(0,1)$ against $\theta_{\text{true}}$.
* **Track Estimators:** Loop through each step, update $\alpha_k$ and $\beta_k$ analytically, and store the computed values for $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$.
* **Visualize:** Use Plotly to create a single line chart showing the progression of both estimators from step $0$ to $100$. Add a static horizontal reference line at $y = 0.35$ representing $\theta_{\text{true}}$.
* **Analysis:** Explain how the distance between your estimators and $\theta_{\text{true}}$ responds as the sampling size $k$ approaches $100$. What does this imply about the accumulation of evidence over time relative to the choice of the initial prior?

### Visualization of the Beta Distribution

In [19]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

# 1. Generate a dense grid of points over the bounded domain [0, 1]
theta_grid = np.linspace(0, 1, 500)

# 2. Define the three distinct parameter configurations
beta_configs = [
    {"alpha": 1, "beta": 1, "name": "Uninformative State: Beta(1,1)", "color": "gray", "dash": "dash"},
    {"alpha": 2, "beta": 8, "name": "Right-Skewed State: Beta(2,8)", "color": "blue", "dash": "solid"},
    {"alpha": 8, "beta": 2, "name": "Left-Skewed State: Beta(8,2)", "color": "green", "dash": "solid"}
]

# 3. Create the Plotly figure
fig = go.Figure()

for config in beta_configs:
    a = config["alpha"]
    b = config["beta"]

    # Calculate the exact analytical probability density function (PDF)
    pdf_vals = stats.beta.pdf(theta_grid, a, b)

    # Add trace to the plot
    fig.add_trace(go.Scatter(
        x=theta_grid,
        y=pdf_vals,
        mode='lines',
        name=config["name"],
        line=dict(color=config["color"], dash=config["dash"], width=2.5)
    ))

# 4. Customize layout and design
fig.update_layout(
    title={
        'text': "Structural Variations of the Beta(α, β) Probability Density Function",
        'y': 0.93,
        'x': 0.5,
        'xanchor': 'center',
        'yanchor': 'top'
    },
    xaxis_title="Parameter Value (θ)",
    yaxis_title="Probability Density f(θ)",
    xaxis=dict(range=[0, 1], gridcolor='rgba(0,0,0,0.1)'),
    yaxis=dict(gridcolor='rgba(0,0,0,0.1)'),
    template="plotly_white",
    hovermode="x unified",
    legend=dict(
        yanchor="top",
        y=0.95,
        xanchor="center",
        x=0.5,
        bgcolor="rgba(255,255,255,0.7)"
    )
)

# Display the interactive plot
fig.show()

### Sequential Likelihood and Joint History

**1. Single Isolated Response Likelihood**

At any specific impression step $k$, the user interaction $y_k \in \{0, 1\}$ is modeled as an independent Bernoulli trial conditional on the true click probability $\theta$. The mathematical likelihood contribution $L(y_k \mid \theta)$ of this single outcome is expressed as:

$$L(y_k \mid \theta) = \theta^{y_k} (1 - \theta)^{1 - y_k}$$

* If the user clicks ($y_k = 1$), the expression simplifies to $\theta^1 (1-\theta)^0 = \theta$.
* If the user does not click ($y_k = 0$), the expression simplifies to $\theta^0 (1-\theta)^1 = 1 - \theta$.

**2. Joint Likelihood Function for the Running History**

Assuming that individual user interactions are conditionally independent given the underlying conversion parameter $\Theta = \theta$, the joint likelihood function for the entire running history vector $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ is the product of the individual step likelihoods up to time step $k$:

$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^k \theta^{y_i} (1 - \theta)^{1 - y_i} = \theta^{\sum_{i=1}^k y_i} (1 - \theta)^{\sum_{i=1}^k (1 - y_i)}$$

Letting $C_k = \sum_{i=1}^k y_i$ represent the total running count of clicks, and $N_k - C_k = \sum_{i=1}^k (1 - y_i)$ represent the total running count of non-clicks (where $N_k = k$ is the total number of impressions), the joint history likelihood simplifies compactly to:

$$L(\mathbf{y}^{(k)} \mid \theta) = \theta^{C_k} (1 - \theta)^{k - C_k}$$

---

**Closed-Form Analytical Updates (Conjugacy)**

**1. Recursive Relationship via Bayes' Theorem**

In a running update framework, the posterior distribution at step $k-1$ acts directly as the prior distribution for step $k$. Applying Bayes' Theorem sequentially, the posterior density after observing the newest single event $y_k$ is:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) = \frac{L(y_k \mid \theta) \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})}{\int_{0}^{1} L(y_k \mid s) \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(s \mid \mathbf{y}^{(k-1)})\,ds}$$

Dropping the denominator (the local normalizing constant) yields the recursive formulation up to a proportionality constant:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto L(y_k \mid \theta) \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$

**2. Analytical Proof of Beta-Binomial Conjugacy**

Assume that the prior density at the start of step $k$ (which is the posterior inherited from step $k-1$) belongs to the Beta family with shape parameters $\alpha_{k-1}$ and $\beta_{k-1}$:

$$f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)}) = \frac{1}{\mathrm{B}(\alpha_{k-1}, \beta_{k-1})} \theta^{\alpha_{k-1} - 1} (1 - \theta)^{\beta_{k-1} - 1} \propto \theta^{\alpha_{k-1} - 1} (1 - \theta)^{\beta_{k-1} - 1}$$

Substituting the single isolated response likelihood from Task 2 into the proportional recursive framework:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \left[ \theta^{y_k} (1 - \theta)^{1 - y_k} \right] \cdot \left[ \theta^{\alpha_{k-1} - 1} (1 - \theta)^{\beta_{k-1} - 1} \right]$$

Combining like algebraic bases by summing their exponential powers:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \theta^{(\alpha_{k-1} + y_k) - 1} \cdot (1 - \theta)^{(\beta_{k-1} + 1 - y_k) - 1}$$

Because the resulting kernel $\theta^{\text{power}_1 - 1}(1-\theta)^{\text{power}_2 - 1}$ matches the exact structural form of a Beta probability density function, the posterior is proven to remain inside the Beta family.

**3. Closed-Form Arithmetic Update Parameters**

Equating the exponent powers directly reveals that the new analytical state parameters $(\alpha_k, \beta_k)$ are simple deterministic linear updates of the previous state:

$$\alpha_k = \alpha_{k-1} + y_k$$

$$\beta_k = \beta_{k-1} + (1 - y_k)$$

Including the exact beta normalization constant, the explicit closed-form posterior density at step $k$ is:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) = \frac{1}{\mathrm{B}(\alpha_k, \beta_k)} \theta^{\alpha_k - 1} (1 - \theta)^{\beta_k - 1}$$

### Posterior Mean


In the context of the Beta-Binomial conjugate update problem, it has a clean, exact analytical solution. Since the posterior distribution at step $k$ is $\text{Beta}(\alpha_k, \beta_k)$, the expected value is given by the standard formula for a Beta distribution's mean:

$$\mathbb{E}[\Theta \mid \mathbf{Y}^{(k)}=\mathbf{y}^{(k)}] = \frac{\alpha_k}{\alpha_k + \beta_k}$$

### Expanding via the Closed-Form Updates

Using the recursive arithmetic update rules we derived in Task 3:

* $\alpha_k = \alpha_0 + \sum_{i=1}^k y_i = \alpha_0 + C_k$ *(where $C_k$ is the total number of clicks observed up to step $k$)*
* $\beta_k = \beta_0 + \sum_{i=1}^k (1 - y_i) = \beta_0 + (k - C_k)$ *(where $k - C_k$ is the total number of non-clicks)*

Substituting these updates directly into the expectation formula gives:

$$\mathbb{E}[\Theta \mid \mathbf{Y}^{(k)}=\mathbf{y}^{(k)}] = \frac{\alpha_0 + C_k}{(\alpha_0 + \beta_0) + k}$$

**Interpretation of the Result:**

This closed-form solution has a beautiful intuitive structure in data analysis:

1. **Weighted Average:** The posterior mean behaves as a weighted average combining the initial prior beliefs ($\alpha_0, \beta_0$) with the real-world sample evidence ($C_k$ successes out of $k$ total impressions).
2. **Asymptotic Convergence:** As the number of impressions grows very large ($k \to \infty$), the constants from the prior ($\alpha_0$ and $\beta_0$) become mathematically insignificant. The formula collapses directly to $\frac{C_k}{k}$, which is the empirical sample proportion of clicks (the maximum likelihood estimate).

## Task 4: Dynamic Shifting Mechanics

At step $k-1$, suppose the posterior is

$$
\Theta\mid\mathbf{Y}^{(k-1)}=\mathbf{y}^{(k-1)}
\sim \operatorname{Beta}(\alpha_{k-1},\beta_{k-1}).
$$

After the new observation $y_k$, the parameters become

$$
\alpha_k=\alpha_{k-1}+y_k,
\qquad
\beta_k=\beta_{k-1}+1-y_k.
$$

For $\alpha_k>1$ and $\beta_k>1$, the posterior mode is

$$
m_k=\frac{\alpha_k-1}{\alpha_k+\beta_k-2}.
$$

If a click is observed, $y_k=1$, then $\alpha_k=\alpha_{k-1}+1$ while $\beta_k=\beta_{k-1}$. The likelihood contribution is $\theta$, which gives relatively more weight to larger values of $\theta$. Consequently, the posterior density and its mode shift to the right.

If a non-click is observed, $y_k=0$, then $\beta_k=\beta_{k-1}+1$ while $\alpha_k=\alpha_{k-1}$. The likelihood contribution is $1-\theta$, which gives relatively more weight to smaller values of $\theta$. Consequently, the posterior density and its mode shift to the left.

This update is available in closed form because the Beta prior is conjugate to the Bernoulli likelihood. In contrast, a normal prior combined with a 2PL item-response likelihood does not produce a standard posterior family. That non-conjugate posterior must normally be evaluated and normalized using numerical integration, a grid method, MCMC, or another approximation.




## Task 5: Running Point Estimators


Since the step-$k$ posterior is

$$
\Theta\mid\mathbf{Y}^{(k)}=\mathbf{y}^{(k)}
\sim\operatorname{Beta}(\alpha_k,\beta_k),
$$

the running posterior mean is

$$
\widehat{\theta}_{\mathrm{Bayes}}^{(k)}
=\mathbb{E}[\Theta\mid\mathbf{Y}^{(k)}=\mathbf{y}^{(k)}]
=\frac{\alpha_k}{\alpha_k+\beta_k}.
$$

If $\alpha_k>1$ and $\beta_k>1$, the unique interior MAP estimate is

$$
\widehat{\theta}_{\mathrm{MAP}}^{(k)}
=\frac{\alpha_k-1}{\alpha_k+\beta_k-2}.
$$

Boundary cases must be handled separately:

$$
\widehat{\theta}_{\mathrm{MAP}}^{(k)}=
\begin{cases}
0, & \alpha_k\leq 1,\ \beta_k>1,\\
1, & \alpha_k>1,\ \beta_k\leq 1,\\
\text{not unique on }[0,1], & \alpha_k=\beta_k=1.
\end{cases}
$$

For the initial uniform prior $\operatorname{Beta}(1,1)$, every value in $[0,1]$ is a mode. A program may use $0.5$ as a convenient representative value at step $0$.

### Numerical Simulation

In [18]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

# Set random seed for reproducibility
np.random.seed(42)

# =====================================================================
# CONFIGURATION & PARAMETERS
# =====================================================================
theta_true = 0.35  # The true click-through rate of the advertisement (35%)
n_impressions = 100
steps = list(range(n_impressions + 1))

# Initialize Beta Prior Parameters (Prior: Uniform distribution Beta(1,1))
alpha_param = 1
beta_param = 1

# Setup grid for plotting probability density curves (Domain: 0 to 1)
theta_grid = np.linspace(0, 1, 500)

# Define specific milestone steps where we want to capture the full curve shape
milestones = [0, 1, 2, 5, 10, 30, 50, 100]

# Tracking arrays for point estimates
running_bayes = [alpha_param / (alpha_param + beta_param)]
running_map = [0.0]  # Mode of Uniform(0,1)

# Initialize Figure 1 for the probability density progression curves
fig1 = go.Figure()

# Plot the initial Prior Beta(1, 1) distribution curve
prior_density = stats.beta.pdf(theta_grid, alpha_param, beta_param)
fig1.add_trace(go.Scatter(
    x=theta_grid, y=prior_density, mode='lines',
    name='Initial Prior: Beta(1,1)',
    line=dict(dash='dash', width=2.5, color='gray')
))

# =====================================================================
# RUNNING ANALYTICAL UPDATE LOOP
# =====================================================================
for k in range(1, n_impressions + 1):
    # Simulate a user action stochastically based on true CTR
    y_k = 1 if np.random.uniform(0, 1) < theta_true else 0

    # EXACT CLOSED-FORM CONJUGATE UPDATE RULES:
    alpha_param += y_k
    beta_param += (1 - y_k)

    # Calculate exact point estimates directly from analytical formulas
    theta_bayes_k = alpha_param / (alpha_param + beta_param)

    # Guard formula for mode when alpha or beta <= 1
    if alpha_param > 1 and beta_param > 1:
        theta_map_k = (alpha_param - 1) / (alpha_param + beta_param - 2)
    else:
        theta_map_k = 0.0 if alpha_param <= beta_param else 1.0

    running_bayes.append(theta_bayes_k)
    running_map.append(theta_map_k)

    # If the current step hits one of our milestone checkpoints, capture its PDF
    if k in milestones:
        # Evaluate exact density curve across the grid analytically
        density_k = stats.beta.pdf(theta_grid, alpha_param, beta_param)
        result_text = "Click" if y_k == 1 else "No Click"

        fig1.add_trace(go.Scatter(
            x=theta_grid, y=density_k, mode='lines',
            name=f"Step {k}: After Event ({result_text}, α={alpha_param}, β={beta_param})",
            line=dict(width=2)
        ))

# =====================================================================
# VISUALIZE FIGURE 1: POSTERIOR DISTRIBUTION PROGRESSION
# =====================================================================
# Add True CTR vertical line to see where the density curves are peaking
fig1.add_vline(
    x=theta_true, line_dash="dot", line_color="red", line_width=2,
    annotation_text=f"True CTR ({theta_true})", annotation_position="top right"
)

fig1.update_layout(
    title={
        'text': "Analytical Posterior Density Progression (Beta-Binomial Updates)",
        'y': 0.95, 'x': 0.5, 'xanchor': 'center', 'yanchor': 'top'
    },
    xaxis_title="Conversion Rate Parameter (θ)",
    yaxis_title="Probability Density f(θ | y)",
    template="plotly_white",
    hovermode="x unified",
    legend=dict(
        yanchor="top", y=0.95, xanchor="right", x=0.98,
        bgcolor="rgba(255,255,255,0.7)"
    )
)
fig1.show()

# =====================================================================
# VISUALIZE FIGURE 2: CONVERGENCE TIMELINE
# =====================================================================
fig2 = go.Figure()

# Add True CTR Reference Line
fig2.add_hline(
    y=theta_true, line_dash="dash", line_color="red", line_width=2,
    annotation_text=f"True CTR (θ = {theta_true})", annotation_position="bottom right"
)

# Add Posterior Mean Trace
fig2.add_trace(go.Scatter(
    x=steps, y=running_bayes, mode='lines',
    name='Exact Posterior Mean (Beta Formula)',
    line=dict(color='blue', width=2.5)
))

# Add MAP Trace
fig2.add_trace(go.Scatter(
    x=steps, y=running_map, mode='lines',
    name='Exact MAP Estimate (Beta Formula)',
    line=dict(color='green', width=1.5, dash='dot')
))

fig2.update_layout(
    title={
        'text': "Analytical Beta-Binomial Conjugate Update Timeline",
        'y': 0.93, 'x': 0.5, 'xanchor': 'center', 'yanchor': 'top'
    },
    xaxis_title="Number of User Impressions (k)",
    yaxis_title="Estimated Conversion Rate (θ̂)",
    template="plotly_white",
    hovermode="x unified",
    legend=dict(yanchor="bottom", y=0.05, xanchor="right", x=0.98)
)
fig2.show()

## Task 6: Convergence Analysis

With the initial prior $\operatorname{Beta}(1,1)$, after $k$ impressions and $C_k=\sum_{i=1}^k y_i$ clicks, the posterior is

$$
\Theta\mid\mathbf{Y}^{(k)}=\mathbf{y}^{(k)}
\sim\operatorname{Beta}(1+C_k,1+k-C_k).
$$

The posterior mean is therefore

$$
\widehat{\theta}_{\mathrm{Bayes}}^{(k)}
=\frac{1+C_k}{k+2},
$$

and, once both shape parameters exceed one, the MAP estimate is

$$
\widehat{\theta}_{\mathrm{MAP}}^{(k)}
=\frac{C_k}{k}.
$$

As $k$ approaches $100$, the empirical click proportion $C_k/k$ generally approaches the true click probability $\theta_{\mathrm{true}}=0.35$. Consequently, both estimators tend to move closer to $0.35$. Their distances from the true value need not decrease at every individual step because the Bernoulli observations are random. Short-term fluctuations are therefore expected.

The variance of a $\operatorname{Beta}(\alpha_k,\beta_k)$ posterior is

$$
\operatorname{Var}(\Theta\mid\mathbf{Y}^{(k)})
=\frac{\alpha_k\beta_k}
{(\alpha_k+\beta_k)^2(\alpha_k+\beta_k+1)}.
$$

Because $\alpha_k+\beta_k=\alpha_0+\beta_0+k$, the posterior variance generally decreases as more observations accumulate. The posterior distribution becomes narrower, indicating greater confidence in the estimated CTR. The initial prior has a noticeable effect when $k$ is small, but its two pseudo-observations become relatively unimportant compared with the evidence from approximately $100$ actual impressions.


# Q Bayesian Estimations for Structural Health Monitoring via Bounded Grid Updates

In aerospace and civil engineering, Structural Health Monitoring (SHM) is critical for detecting damage before a catastrophic failure occurs. Consider an aircraft wing or a bridge girder equipped with specialized vibration sensors. Over time, environmental fatigue or dynamic impacts can cause micro-fractures, resulting in a reduction of the component's mechanical stiffness.

Let $\Theta = \theta$ represent the structural **remaining stiffness efficiency factor**, where $\theta$ is physically bounded to the interval:

$$\theta \in (0, 1]$$

* $\theta = 1.0$ indicates a perfectly pristine, undamaged structural component.
* $\theta \to 0$ signifies critical degradation or severe structural cracking.

Let $K_{\text{nominal}}$ be the known, baseline stiffness of the structural component when it is entirely healthy. At each sequential inspection time step $k$ (where $k = 1, 2, \dots, n$), a sensor collects a noisy experimental stiffness measurement $y_k$.

Engineers model the degradation physics via a non-linear relationship with multiplicative log-normal measurement noise to prevent non-physical negative values:

$$y_k = \theta \cdot K_{\text{nominal}} \cdot e^{\epsilon_k}, \qquad \epsilon_k \sim \mathscr{N}(0, \sigma^2)$$

where $\sigma$ is the standard deviation of the sensor noise in log-space.

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running history vector of observed sensor readings** up to the current inspection milestone. Before deploying the sensors, engineers utilize an initial prior distribution $f_{\Theta}^{(0)}(\theta)$ over the domain $(0, 1]$ based on historical manufacturing specifications. As the sensor stream arrives, the posterior distribution calculated at step $k-1$ serves directly as the prior distribution for step $k$.

---

### **Tasks**

#### **1. Prior Belief Boundaries**

Before data collection begins, engineers assume the component is highly likely to be healthy, modeling this using a bounded Beta distribution as the initial prior: $\Theta \sim \text{Beta}(8, 1.5)$.

* Plot this initial prior density function using Plotly over the restricted physical domain $\theta \in [0.01, 1.0]$.
* Calculate the expected prior stiffness efficiency $\mathbb{E}[\Theta^{(0)}]$ analytically. Explain why this specific distribution serves as an appropriate initial prior for an engineering component assumed to be healthy.

#### **2. Structural Likelihood Formulation**

Using the change of variables or properties of the log-normal distribution, write down the mathematical likelihood contribution $L(y_k \mid \theta)$ of a *single* continuous sensor measurement $y_k$ at inspection step $k$, given the true stiffness factor $\theta$. Following this, write down the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.

#### **3. Mathematical Formulation of the Non-Conjugate Grid Update**

Explain why an exact closed-form analytical solution for the posterior density $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$ does not exist when combining a Beta prior with this log-normal structural likelihood. Write down the recursive relationship for the posterior density at step $k$ up to a proportionality constant.

#### **4. Running Point Estimates**

Because a closed-form formula is unavailable, we must define point estimators through numerical integration. Write down the definite integral equations over the bounded domain $(0, 1]$ required to compute:

* The **Running Posterior Mean** ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$)
* The **Running Maximum A Posteriori** ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$)

#### **5. Algorithmic Grid Approximation and Normalization**

Describe the step-by-step numerical procedure to maintain this distribution on a discrete grid of $\theta$-values. Explicitly state how you would handle the boundary limits computationally and how you would perform the sequential normalization step using the trapezoidal rule after a new sensor reading $y_k$ is observed.

#### **6. Performance Tracking and Degradation Convergence Analysis**

Suppose an impact occurs, and the true, hidden remaining stiffness drops to $\theta_{\text{true}} = 0.68$. Write a Python script using Plotly to simulate an engineered monitoring timeline across $n = 15$ continuous sensor measurements ($K_{\text{nominal}} = 50.0 \text{ kN/mm}$, $\sigma = 0.15$):

* **Simulate Sensor Stream:** Programmatically generate noisy sensor readings $y_k$ by drawing random values from the underlying log-normal physics model centered at $\theta_{\text{true}}$.
* **Track Estimators:** Loop sequentially through each step. At each step, update the unnormalized grid, normalize it via `np.trapezoid`, and compute both $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$.
* **Visualize Curves & Timeline:** Generate two plots:
1. A plot showing the progression of the full posterior density curves at milestones $k \in \{0, 1, 2, 5, 10, 15\}$.
2. A line chart tracking the convergence of both $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$ from step $0$ to $15$ against a horizontal reference line at $\theta_{\text{true}} = 0.68$.


* **Analysis:** Evaluate the behavior of the distribution. How many sensor readings did it take for the system to overcome the initially optimistic "healthy" prior and confidently isolate the 68% damage state? What does the narrowing of the density curves imply about structural safety thresholds?

## Sample Answer

1. Physical Likelihood Formulation

At time step $k$, a physical sensor records a continuous measurement $Y_k = y_k$ (such as dynamic modal frequency, strain measurement, or vibrational response).

The measurement model is governed by a known structural forward response function $g(\theta)$ subject to additive Gaussian sensor noise $\epsilon_k \sim \mathscr{N}(0, \sigma^2)$:

$$Y_k = g(\theta) + \epsilon_k$$

Conditional on the underlying structural integrity parameter $\Theta = \theta$, the likelihood contribution of a single observation $y_k$ at step $k$ is given by the Gaussian probability density function:

$$L(y_k \mid \theta) = f_{Y_k \mid \Theta}(y_k \mid \theta) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left( -\frac{\left(y_k - g(\theta)\right)^2}{2\sigma^2} \right)$$

---

2. Sequential Likelihood and Joint History

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)^T$ represent the vector of sequential sensor observations gathered up to step $k$.

Assuming that sensor noise terms across consecutive time steps are conditionally independent given $\Theta = \theta$, the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$ is:

$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^k L(y_i \mid \theta) = \frac{1}{(2\pi\sigma^2)^{k/2}} \exp\left( -\frac{1}{2\sigma^2} \sum_{i=1}^k \left(y_i - g(\theta)\right)^2 \right)$$

---

3. Mathematical Formulation of Bounded Recursive Updates

Before observing measurements, the platform initializes a prior density function $f_{\Theta}^{(0)}(\theta)$ over the physically bounded interval $[\theta_{\min}, \theta_{\max}]$ (e.g., a uniform distribution $U(\theta_{\min}, \theta_{\max})$ indicating initial uninformative uncertainty).

Under a sequential Bayesian updating framework, the posterior distribution at step $k-1$ serves as the prior distribution for step $k$. The running posterior density function $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$ is recursively updated via Bayes' Theorem:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) = \frac{L(y_k \mid \theta) \, f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})}{\int_{\theta_{\min}}^{\theta_{\max}} L(y_k \mid s) \, f_{\Theta \mid \mathbf{Y}^{(k-1)}}(s \mid \mathbf{y}^{(k-1)}) \, ds}$$

*Definition of Key Components*:

* **$f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$**: The prior density at step $k$ inherited directly from the previous state $k-1$.
* **$L(y_k \mid \theta)$**: The likelihood contribution of the incoming real-time sensor reading $y_k$.
* **Denominator (Normalizing Constant $Z_k$)**: Integrates the product of likelihood and prior across the bounded domain $[\theta_{\min}, \theta_{\max}]$ to ensure the total area under the density curve equals $1$.

---

4. Running Point Estimators

From the running posterior distribution $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$, two primary estimators track structural health at step $k$:

a. Running Posterior Mean ($\widehat{\theta}_{\text{Bayes}}^{(k)}$)
Under a squared-error loss function, the optimal point estimate is the expected value of the current bounded posterior distribution:

$$\widehat{\theta}_{\text{Bayes}}^{(k)} = \mathbb{E}\left[\Theta \mid \mathbf{Y}^{(k)} = \mathbf{y}^{(k)}\right] = \int_{\theta_{\min}}^{\theta_{\max}} \theta \, f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \, d\theta$$

b.. Running Maximum A Posteriori ($\widehat{\theta}_{\text{MAP}}^{(k)}$)
The most probable structural state corresponds to the peak (mode) of the current posterior density over the bounded domain:

$$\widehat{\theta}_{\text{MAP}}^{(k)} = \arg\max_{\theta \in [\theta_{\min}, \theta_{\max}]} f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$$

---

5. Numerical Implementation via Bounded Grid Discretization

Since non-linear structural response functions $g(\theta)$ generally lack closed-form analytical conjugate solutions, the system maintains the posterior on a fine discrete grid across the bounded physical domain $[\theta_{\min}, \theta_{\max}]$.

*Algorithmic Procedure*:

a. **Grid Setup:** Define $M$ equally spaced grid points over $[\theta_{\min}, \theta_{\max}]$:
   $$\theta_m = \theta_{\min} + (m-1)\Delta\theta, \quad \text{where } \Delta\theta = \frac{\theta_{\max} - \theta_{\min}}{M-1}, \quad m = 1, 2, \dots, M$$

b. **Prior Initialization:** Evaluate initial prior values across the grid array $\mathbf{P}_0 = [P_0(\theta_1), \dots, P_0(\theta_M)]$ and normalize using numerical integration (e.g., composite trapezoidal rule):
   $$Z_0 = \text{trapezoid}(\mathbf{P}_0, \boldsymbol{\theta}), \quad \mathbf{P}_0 \leftarrow \frac{\mathbf{P}_0}{Z_0}$$

c. **Sequential Updating & Normalization (at step $k$):**
   * Compute unnormalized posterior array: $\tilde{P}_k(\theta_m) = P_{k-1}(\theta_m) \times L(y_k \mid \theta_m)$
   * Compute normalizing factor: $Z_k = \sum_{m=1}^{M-1} \frac{\tilde{P}_k(\theta_m) + \tilde{P}_k(\theta_{m+1})}{2} \Delta\theta$
   * Normalize: $P_k(\theta_m) = \frac{\tilde{P}_k(\theta_m)}{Z_k}$

d. **Point Estimation Evaluation:**
   * **Bayes Estimate:** $\widehat{\theta}_{\text{Bayes}}^{(k)} \approx \text{trapezoid}(\boldsymbol{\theta} \odot \mathbf{P}_k, \boldsymbol{\theta})$
   * **MAP Estimate:** $\widehat{\theta}_{\text{MAP}}^{(k)} = \theta_{m^*}, \quad \text{where } m^* = \arg\max_{m} P_k(\theta_m)$

---

6. Dynamic Mechanics & Convergence Interpretation

* **Variance Reduction:** As $k$ increases, repeated noisy sensor measurements progressively attenuate likelihood ambiguity, tightening the posterior distribution variance around the true structural state $\theta_{\text{true}}$.
* **Physical Boundary Enforcement:** The bounded grid framework strictly guarantees that probability mass outside $[\theta_{\min}, \theta_{\max}]$ is zero, avoiding physically unfeasible state estimates (such as negative stiffness or over-100% health).
* **Sensor Noise vs. Tracking Sensitivity:** Lower measurement noise $\sigma$ narrows the single-step likelihood curve $L(y_k \mid \theta)$, allowing rapid posterior convergence with fewer sensor readings.

In [17]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

# Set random seed for reproducibility
np.random.seed(24)

# =====================================================================
# CONFIGURATION & PARAMETERS
# =====================================================================
theta_true = 0.68       # True remaining stiffness efficiency of the beam (68%)
K_nominal = 50.0        # Nominal baseline stiffness of the pristine structure (kN/mm)
sigma = 0.15            # Sensor noise standard deviation (log-space)
n_sensor_readings = 15  # Timeline steps

# 1. Define a fine grid over the physical boundary [0.01, 1.0]
theta_grid = np.linspace(0.01, 1.0, 500)

# 2. Initialize Prior: Bounded Beta distribution reflecting an initially healthy beam
# centered heavily near 0.95-1.0
current_posterior = stats.beta.pdf(theta_grid, a=8, b=1.5)
# Normalize initial prior
current_posterior /= np.trapezoid(current_posterior, theta_grid)

# Steps milestone tracking for plotting curves
milestones = [0, 1, 2, 5, 10, 15]

# Create Figure
fig = go.Figure()

# Plot Initial Prior State
fig.add_trace(go.Scatter(
    x=theta_grid, y=current_posterior, mode='lines',
    name='Prior State: Structural Health Assumed Healthy',
    line=dict(dash='dash', width=2.5, color='gray')
))

# =====================================================================
# SEQUENTIAL BAYESIAN MONITORING LOOP
# =====================================================================
for k in range(1, n_sensor_readings + 1):
    # Simulate a noisy structural sensor reading from log-normal physics
    noise = np.random.normal(0, sigma)
    y_k = (theta_true * K_nominal) * np.exp(noise)

    # Calculate Log-Normal Likelihood curve across the structural theta grid
    # Expected value for any grid point is: grid_point * K_nominal
    expected_K = theta_grid * K_nominal
    likelihood = stats.lognorm.pdf(y_k, s=sigma, scale=expected_K)

    # Running Update: Posterior Proportional to Prior * Likelihood
    current_posterior = current_posterior * likelihood

    # Numerical Normalization via Trapezoidal rule
    integral = np.trapezoid(current_posterior, theta_grid)
    current_posterior /= integral

    # Capture structural health density profile at milestones
    if k in milestones:
        fig.add_trace(go.Scatter(
            x=theta_grid, y=current_posterior, mode='lines',
            name=f"Step {k}: Post-Sensor Reading (Observed K={y_k:.2f})",
            line=dict(width=2)
        ))

# =====================================================================
# VISUALIZE STRUCTURAL DEGRADATION TRACKING
# =====================================================================
fig.add_vline(
    x=theta_true, line_dash="dot", line_color="red", line_width=2.5,
    annotation_text=f"True Structural Degradation State ({theta_true})",
    annotation_position="top left"
)

fig.update_layout(
    title={
        'text': "Structural Health Monitoring: Bounded Bayesian Parameter Updating",
        'y': 0.95, 'x': 0.5, 'xanchor': 'center', 'yanchor': 'top'
    },
    xaxis_title="Remaining Structural Stiffness Efficiency Factor (θ)",
    yaxis_title="Probability Density (Confidence level of damage)",
    template="plotly_white",
    hovermode="x unified",
    legend=dict(
        yanchor="top", y=0.95, xanchor="left", x=0.02,
        bgcolor="rgba(255,255,255,0.7)"
    )
)

fig.show()

# Answer for Q3

## 1. Prior Belief Boundaries

The initial belief is

$$
\Theta\sim\operatorname{Beta}(8,1.5),
$$

with density

$$
f_\Theta^{(0)}(\theta)
=\frac{1}{\mathrm{B}(8,1.5)}\theta^{8-1}(1-\theta)^{1.5-1},
\qquad 0<\theta<1.
$$

For a $\operatorname{Beta}(\alpha,\beta)$ distribution,

$$
\mathbb{E}[\Theta]=\frac{\alpha}{\alpha+\beta}.
$$

Therefore,

$$
\mathbb{E}[\Theta^{(0)}]
=\frac{8}{8+1.5}
=\frac{8}{9.5}
\approx0.8421.
$$

The prior assigns most probability to relatively high stiffness values, expressing the engineering belief that a newly inspected component is likely to be healthy. It remains uncertain rather than placing all mass at $\theta=1$, so moderate degradation is still possible before sensor evidence arrives. The support $(0,1)$ also respects the physical stiffness-efficiency bounds.


In [16]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta

theta_grid = np.linspace(0.01, 1.0, 4000)
prior = beta.pdf(theta_grid, a=8, b=1.5)
prior /= np.trapezoid(prior, theta_grid)
prior_mean = 8 / (8 + 1.5)

print(f"Analytical prior mean = {prior_mean:.4f}")

fig_prior = go.Figure()
fig_prior.add_trace(go.Scatter(
    x=theta_grid, y=prior, mode="lines",
    name="Beta(8, 1.5)", line=dict(width=3)
))
fig_prior.add_vline(
    x=prior_mean, line_dash="dash", line_color="red",
    annotation_text=f"Prior mean = {prior_mean:.4f}"
)
fig_prior.update_layout(
    title="Initial Prior for Remaining Stiffness Efficiency",
    xaxis_title="Remaining stiffness efficiency, θ",
    yaxis_title="Prior density",
    template="plotly_white"
)
fig_prior.show()


Analytical prior mean = 0.8421


## 2. Structural Likelihood Formulation

The measurement model is

$$
Y_k=\theta K_{\mathrm{nominal}}e^{\epsilon_k},
\qquad \epsilon_k\sim\mathcal{N}(0,\sigma^2).
$$

Taking logarithms gives

$$
\log Y_k
=\log(\theta K_{\mathrm{nominal}})+\epsilon_k.
$$

Hence,

$$
\log Y_k\mid\Theta=\theta
\sim\mathcal{N}\!\left(\log(\theta K_{\mathrm{nominal}}),\sigma^2\right),
$$

so $Y_k\mid\Theta=\theta$ is log-normal. The likelihood contribution of one positive reading $y_k$ is

$$
L(y_k\mid\theta)
=\frac{1}{y_k\sigma\sqrt{2\pi}}
\exp\!\left\{-\frac{[\log y_k-\log(\theta K_{\mathrm{nominal}})]^2}{2\sigma^2}\right\},
\qquad y_k>0.
$$

Assuming conditional independence, the joint likelihood is

$$
L(\mathbf{y}^{(k)}\mid\theta)
=\prod_{i=1}^{k}L(y_i\mid\theta)
$$

or explicitly,

$$
L(\mathbf{y}^{(k)}\mid\theta)
=\frac{1}{(\sigma\sqrt{2\pi})^k\prod_{i=1}^{k}y_i}
\exp\!\left\{-\frac{1}{2\sigma^2}
\sum_{i=1}^{k}[\log y_i-\log(\theta K_{\mathrm{nominal}})]^2\right\}.
$$


## 3. Non-Conjugate Recursive Update

Multiplying the Beta prior kernel by the log-normal likelihood produces terms involving both powers of $\theta$ and squared logarithms of $\theta$. The result cannot be rewritten as the kernel of a Beta distribution or another convenient standard family. Thus, the Beta prior is not conjugate to this likelihood and no simple closed-form sequence of posterior parameters is available.

The recursive update is

$$
f_k(\theta)
\equiv
f_{\Theta\mid\mathbf{Y}^{(k)}}(\theta\mid\mathbf{y}^{(k)})
\propto L(y_k\mid\theta)f_{k-1}(\theta),
\qquad 0<\theta\leq1.
$$

Its normalized form is

$$
f_k(\theta)
=\frac{L(y_k\mid\theta)f_{k-1}(\theta)}
{\displaystyle\int_0^1L(y_k\mid s)f_{k-1}(s)\,ds}.
$$

At $k=0$,

$$
f_0(\theta)=\frac{\theta^7(1-\theta)^{0.5}}{\mathrm{B}(8,1.5)}.
$$


## 4. Running Point Estimates

The running posterior mean is

$$
\widehat\theta_{\mathrm{Bayes}}^{(k)}
=\mathbb{E}[\Theta\mid\mathbf{Y}^{(k)}=\mathbf{y}^{(k)}]
=\int_0^1\theta f_k(\theta)\,d\theta.
$$

The running MAP estimate is

$$
\widehat\theta_{\mathrm{MAP}}^{(k)}
\in\underset{0<\theta\leq1}{\operatorname{argmax}}\;f_k(\theta).
$$

On a numerical grid, these become

$$
\widehat\theta_{\mathrm{Bayes}}^{(k)}
\approx\operatorname{Trapz}(\theta_gf_k(\theta_g),\theta_g)
$$

and

$$
\widehat\theta_{\mathrm{MAP}}^{(k)}
\approx\theta_{\operatorname{argmax}_g f_k(\theta_g)}.
$$


## 5. Algorithmic Grid Approximation and Normalization

Because the log-normal likelihood contains $\log\theta$, the value $\theta=0$ cannot be used computationally. Define a small positive lower boundary $\varepsilon$, such as $0.01$, and construct

$$
\theta_g\in[\varepsilon,1],\qquad g=1,\ldots,G.
$$

The approximation assumes zero probability outside the physical interval $(0,1]$.

1. Evaluate the $\operatorname{Beta}(8,1.5)$ prior at each grid point.
2. Normalize the initial grid using the trapezoidal rule.
3. After observing $y_k$, evaluate the log-normal likelihood $L(y_k\mid\theta_g)$ at every grid point.
4. Form the unnormalized update

$$
\widetilde f_k(\theta_g)=L(y_k\mid\theta_g)f_{k-1}(\theta_g).
$$

5. Calculate

$$
Z_k\approx\operatorname{Trapz}(\widetilde f_k(\theta_g),\theta_g).
$$

6. Normalize:

$$
f_k(\theta_g)=\frac{\widetilde f_k(\theta_g)}{Z_k}.
$$

7. Compute the posterior mean, MAP, variance, and any credible interval from the normalized grid.

A fine grid reduces discretization error. The range must include all physically possible states, and its lower endpoint must remain strictly positive.


In [15]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.stats import beta, lognorm

rng = np.random.default_rng(24)

theta_true = 0.68
K_nominal = 50.0
sigma = 0.15
n_readings = 15
theta_grid = np.linspace(0.01, 1.0, 4000)
milestones = {0, 1, 2, 5, 10, 15}

# Initial Beta(8, 1.5) prior
posterior = beta.pdf(theta_grid, a=8, b=1.5)
posterior /= np.trapezoid(posterior, theta_grid)

steps = [0]
readings = [np.nan]
means = [np.trapezoid(theta_grid * posterior, theta_grid)]
maps = [theta_grid[np.argmax(posterior)]]
variances = [np.trapezoid((theta_grid-means[0])**2 * posterior, theta_grid)]
lower_95 = [np.nan]
upper_95 = [np.nan]
saved_posteriors = {0: posterior.copy()}

def grid_quantile(grid, density, probability):
    increments = 0.5 * (density[:-1] + density[1:]) * np.diff(grid)
    cdf = np.concatenate(([0.0], np.cumsum(increments)))
    cdf /= cdf[-1]
    return np.interp(probability, cdf, grid)

for k in range(1, n_readings + 1):
    epsilon_k = rng.normal(0.0, sigma)
    y_k = theta_true * K_nominal * np.exp(epsilon_k)

    likelihood = lognorm.pdf(
        y_k,
        s=sigma,
        scale=theta_grid * K_nominal
    )

    unnormalized = posterior * likelihood
    normalizer = np.trapezoid(unnormalized, theta_grid)
    if normalizer <= 0 or not np.isfinite(normalizer):
        raise ValueError(f"Normalization failed at step {k}.")
    posterior = unnormalized / normalizer

    mean_k = np.trapezoid(theta_grid * posterior, theta_grid)
    map_k = theta_grid[np.argmax(posterior)]
    var_k = np.trapezoid((theta_grid-mean_k)**2 * posterior, theta_grid)

    steps.append(k)
    readings.append(y_k)
    means.append(mean_k)
    maps.append(map_k)
    variances.append(var_k)
    lower_95.append(grid_quantile(theta_grid, posterior, 0.025))
    upper_95.append(grid_quantile(theta_grid, posterior, 0.975))

    if k in milestones:
        saved_posteriors[k] = posterior.copy()

# Figure 1: full posterior curves
fig_curves = go.Figure()
for k in sorted(saved_posteriors):
    fig_curves.add_trace(go.Scatter(
        x=theta_grid, y=saved_posteriors[k], mode="lines",
        name=f"k = {k}", line=dict(width=2.5)
    ))
fig_curves.add_vline(
    x=theta_true, line_dash="dash", line_color="red",
    annotation_text="True state = 0.68"
)
fig_curves.update_layout(
    title="Sequential Posterior Density Curves",
    xaxis_title="Remaining stiffness efficiency, θ",
    yaxis_title="Posterior density",
    template="plotly_white",
    hovermode="x unified"
)
fig_curves.show()

# Figure 2: estimator convergence
fig_timeline = go.Figure()
fig_timeline.add_trace(go.Scatter(
    x=steps, y=means, mode="lines+markers",
    name="Posterior Mean", line=dict(width=3, color="royalblue")
))
fig_timeline.add_trace(go.Scatter(
    x=steps, y=maps, mode="lines+markers",
    name="MAP", line=dict(width=2.5, color="seagreen")
))
fig_timeline.add_hline(
    y=theta_true, line_dash="dash", line_color="red",
    annotation_text="True state = 0.68"
)
fig_timeline.update_layout(
    title="Convergence of Structural Stiffness Estimators",
    xaxis_title="Number of sensor readings, k",
    yaxis_title="Estimated stiffness efficiency",
    xaxis=dict(dtick=1), template="plotly_white",
    hovermode="x unified"
)
fig_timeline.show()

results = pd.DataFrame({
    "Step": steps,
    "Sensor reading": readings,
    "Posterior mean": means,
    "MAP": maps,
    "Posterior variance": variances,
    "95% lower": lower_95,
    "95% upper": upper_95
})
display(results.round(4))

# A transparent operational criterion for overcoming the optimistic prior:
# mean within 0.05 of truth AND 95% interval excludes the prior mean.
prior_mean = 8 / 9.5
overcome_step = next((
    k for k in range(1, n_readings + 1)
    if abs(means[k] - theta_true) <= 0.05 and upper_95[k] < prior_mean
), None)
print("First step satisfying the stated confidence criterion:", overcome_step)

,Step,Sensor reading,Posterior mean,MAP,Posterior variance,95% lower,95% upper
0,0,NaN,0.8421,0.9334,0.0127,NaN,NaN
1,1,41.6363,0.8594,0.8901,0.0062,0.6878,0.9831
2,2,35.7956,0.8166,0.8148,0.0055,0.6719,0.9575
3,3,28.5573,0.7356,0.7287,0.0039,0.6205,0.8636
4,4,33.0591,0.7171,0.7116,0.0028,0.6184,0.8261
5,5,32.3121,0.7027,0.6982,0.0022,0.6154,0.7984
6,6,32.8585,0.6950,0.6913,0.0018,0.6158,0.7814
7,7,37.1844,0.7017,0.6985,0.0016,0.6273,0.7822
8,8,28.0645,0.6826,0.6799,0.0013,0.6146,0.7559
9,9,39.3053,0.6933,0.6908,0.0012,0.6281,0.7633


First step satisfying the stated confidence criterion: 4


## 6. Performance and Degradation-Convergence Analysis

The result depends on the random sensor stream, so the phrase “overcome the prior” requires an explicit criterion. In the reproducible simulation above, define it as the first step at which:

1. the posterior mean is within $0.05$ of $\theta_{\mathrm{true}}=0.68$; and
2. the upper endpoint of the $95\%$ credible interval is below the optimistic prior mean $8/9.5\approx0.8421$.

With seed $24$, this criterion is first satisfied after approximately **four sensor readings**. This is a simulation-specific result rather than a universal constant. Different random readings or a different confidence rule can change the reported step.

The initial posterior is centered near $0.8421$, reflecting the belief that the component is healthy. Early measurements pull the posterior toward $0.68$. As additional readings accumulate, the likelihood increasingly dominates the initial prior, and both the posterior mean and MAP approach the damage state.

The density curves also become narrower. This means posterior uncertainty is decreasing: the monitoring system is gaining confidence that the reduced stiffness is persistent rather than a single noisy measurement. From a safety perspective, a narrow posterior lying below a maintenance threshold provides stronger evidence for inspection or intervention than a single point estimate. A sound decision rule should therefore use posterior probabilities such as

$$
P(\Theta<\theta_{\mathrm{safety}}\mid\mathbf{y}^{(k)})
=\int_0^{\theta_{\mathrm{safety}}}f_k(\theta)\,d\theta,
$$

rather than relying only on the mean or MAP. A narrow distribution near a threshold also means that small threshold changes can materially affect the operational decision, so the threshold must be chosen using engineering risk and reliability requirements.


# Q. Gaussian Mixture Clustering as Conditional Updating

Consider a dataset
$$
x_1,x_2,\dots,x_n\in\mathbb R^d.
$$
We wish to cluster these observations into $K$ groups. Instead of assigning each point deterministically to a cluster at the beginning, we introduce a latent random variable
$$
C_i\in{1,\dots,K},
$$
where $C_i=k$ means that the observation $x_i$ belongs to cluster $k$.
Let the prior probability of cluster membership be
$$
P(C_i=k)=\phi_k,
$$
where
$$
\phi_k\ge 0,
\qquad
\sum_{k=1}^K \phi_k=1.
$$

Conditional on $C_i=k$, assume that the observation $X_i$ is generated from a multivariate Gaussian distribution:
$$
X_i\mid C_i=k
\sim
\mathscr N(\mu_k,\Sigma_k),
$$
where
$$
\mu_k\in\mathbb R^d,
\qquad
\Sigma_k\in\mathbb R^{d\times d}
$$
are the mean vector and covariance matrix of cluster $k$.

The model parameters
$$
\phi_k,\mu_k,\Sigma_k,
\qquad k=1,\dots,K,
$$
are assumed to be fixed but unknown.

---

1. Deriving the Marginal Density:
Using the law of total probability, show that the marginal density of $X_i$ is
$$
p(x_i)=\sum_{k=1}^K
\phi_k
\mathscr N(x_i\mid \mu_k,\Sigma_k).
$$
Explain why this density is called a Gaussian mixture density.

---

2. Deriving the Posterior Cluster Probability:
For a fixed observation $x_i$, use Bayes' rule to derive
$$
P(C_i=k\mid X_i=x_i)=\frac{
P(X_i=x_i\mid C_i=k)P(C_i=k)
}{
\sum_{j=1}^K P(X_i=x_i\mid C_i=j)P(C_i=j)
}.
$$
Then substitute the Gaussian model and the cluster prior to obtain
$$
P(C_i=k\mid X_i=x_i)=\frac{
\phi_k\mathscr N(x_i\mid \mu_k,\Sigma_k)
}{
\sum_{j=1}^K
\phi_j\mathscr N(x_i\mid \mu_j,\Sigma_j)
}.
$$
This quantity is called the responsibility of cluster $k$ for data point $x_i$, and is denoted by
$$
\gamma_{ik}=P(C_i=k\mid X_i=x_i).
$$
Explain why $\gamma_{ik}$ may be interpreted as a posterior probability of cluster membership.

---

3. One-Hot Encoding of the Latent Cluster Variable:
Now define a one-hot encoded latent random vector
$$
Z_i=
\begin{bmatrix}
Z_{i1}\\
Z_{i2}\\
\vdots\\
Z_{iK}
\end{bmatrix},
$$
where
$$
Z_{ik}=\begin{cases}
1, & \text{if } C_i=k,\\
0, & \text{otherwise}.
\end{cases}
$$
Show that
$$
\mathbb E[Z_{ik}\mid X_i=x_i]=P(C_i=k\mid X_i=x_i).
$$
Hence show that
$$
\mathbb E[Z_i\mid X_i=x_i]=\begin{bmatrix}
\gamma_{i1}\\
\gamma_{i2}\\
\vdots\\
\gamma_{iK}
\end{bmatrix}.
$$
Conclude that the soft cluster assignment in a Gaussian mixture model is precisely the conditional expectation
$$
\mathbb E[Z_i\mid X_i=x_i].
$$

---

4. From Soft Assignment to Hard Clustering:
The vector
$$
\mathbb E[Z_i\mid X_i=x_i]
$$
gives a soft assignment of $x_i$ to all clusters. A hard cluster assignment can be obtained by choosing the cluster with the largest posterior probability:
$$
\widehat C_i=\operatorname{arg\,max}_{1\le k\le K}
\gamma_{ik}.
$$
Explain the difference between soft clustering and hard clustering in this context.

---

5. Conditional Expectation of the Observation Given the Cluster:
Show that
$$
\mathbb E[X_i\mid C_i=k]=\mu_k.
$$
Explain why $\mu_k$ can be interpreted as the center of cluster $k$.
Then compare the two conditional expectations
$$
\mathbb E[Z_i\mid X_i=x_i]
$$
and
$$
\mathbb E[X_i\mid C_i=k].
$$
Explain why the first gives the soft cluster membership of an observed point, while the second gives the mean location of a cluster.

---

6. The Complete-Data Likelihood
If the latent labels $z_i$ were known, the complete-data likelihood would be
$$
p(x_1,\dots,x_n,z_1,\dots,z_n)=\prod_{i=1}^n
\prod_{k=1}^K
\left[
\phi_k
\mathscr N(x_i\mid \mu_k,\Sigma_k)
\right]^{z_{ik}}.
$$
Take the logarithm and show that the complete-data log-likelihood is
$$
\ell_c=\sum_{i=1}^n
\sum_{k=1}^K
z_{ik}
\left[
\log \phi_k
+
\log \mathscr N(x_i\mid \mu_k,\Sigma_k)
\right].
$$
Explain why this expression would be easy to maximize if the values of $z_{ik}$ were known.

---

7. The EM Interpretation:
In practice, the latent variables $Z_i$ are not observed. The EM algorithm replaces the unknown indicators $z_{ik}$ by their conditional expectations given the observed data and current parameter estimates:
$$
z_{ik}
\quad\leadsto\quad
\mathbb E[Z_{ik}\mid X_i=x_i].
$$
That is,
$$
z_{ik}
\quad\leadsto\quad
\gamma_{ik}.
$$
This is the E-step of the EM algorithm.
Using this idea, write the expected complete-data log-likelihood:
$$
Q=\sum_{i=1}^n
\sum_{k=1}^K
\gamma_{ik}
\left[
\log \phi_k
+
\log \mathscr N(x_i\mid \mu_k,\Sigma_k)
\right].
$$
Explain why the E-step can be interpreted as a conditional update of cluster membership probabilities.

---

8. Parameter Updates:
By maximizing $Q$ with respect to the model parameters, derive the standard GMM updates:
$$
N_k=\sum_{i=1}^n \gamma_{ik},
$$
$$
\phi_k^{\text{new}}=\frac{N_k}{n},
$$
$$
\mu_k^{\text{new}}=\frac{1}{N_k}
\sum_{i=1}^n
\gamma_{ik}x_i,
$$
and
$$
\Sigma_k^{\text{new}}=\frac{1}{N_k}
\sum_{i=1}^n
\gamma_{ik}
(x_i-\mu_k^{\text{new}})
(x_i-\mu_k^{\text{new}})^T.
$$
Explain how the responsibility $\gamma_{ik}$ acts as a fractional membership weight of observation $x_i$ in cluster $k$.

---

9. Interpretation:
Write a short paragraph explaining why GMM clustering can be viewed as a repeated process of conditional updating.
Your answer should mention the following points:

* The mixture weight $\phi_k$ is the prior probability of cluster $k$.
* The Gaussian density $\mathscr N(x_i\mid \mu_k,\Sigma_k)$ measures how compatible $x_i$ is with cluster $k$.
* The responsibility $\gamma_{ik}$ is the posterior probability of cluster $k$ after observing $x_i$.
* The soft assignment vector is
$$
\mathbb E[Z_i\mid X_i=x_i].
$$

* The M-step updates the cluster parameters using these posterior membership probabilities as weights.
Conclude that Gaussian mixture clustering is probabilistic clustering based on conditional expectations of latent cluster membership variables.

---

Here is a perfectly tailored question that you can add as the final part (**Part 10**) of your assignment notebook to bridge your theoretical derivations with your code implementation:

---

10. Computational Simulation and Out-of-Sample Validation

Using the theoretical framework established in the previous parts, write a Python class named `GMMFinancialSegmenter` that implements a two-dimensional Gaussian Mixture Model (GMM) using `scikit-learn` and visualizes the results interactively using `Plotly`. Your implementation should fulfill the following criteria:

* **Data Splitting and Scaling:** Accept a dataset containing two continuous features (e.g., mimicking financial behaviors like `PURCHASES` and `CREDIT_LIMIT`), standardize the features to handle variance scaling, and split the data into an 80% training set and a 20% validation/test set.
* **EM Execution:** Fit a GMM with $K=3$ components on the training data using the Expectation-Maximization (EM) algorithm, printing whether the model successfully converged and the number of iterations required.
* **Out-of-Sample Performance:** Compute and output the average log-likelihood score over the unseen test set to validate how well the learned density functions generalize to new data.
* **Interactive Visualizations:** Implement methods to generate three distinct Plotly figures:
1. An empirical **2D Density Heatmap** of the raw training data with marginal distributions to inspect its underlying multimodal structure.
2. A **Training Assignment Plot** that overlays the training data points on top of a continuous contour map showing the maximum posterior responsibilities ($\gamma_{ik}$) computed across a fine coordinate grid.
3. A **Test Assignment Plot** that replicates the contour boundary visualization but overlays out-of-sample test data points to expose the physical regions of cluster ambiguity.



Briefly evaluate the resulting plots. Explain how the continuous background contour map visually demonstrates the soft assignment expectation vector $\mathbb{E}[Z_i \mid X_i = x_{\text{grid}}]$ that you proved analytically in Part 3.

Use the dataset

https://www.kaggle.com/datasets/arjunbhasin2013/ccdata

## Sample Answer

### Part 1

To find the marginal density $p(x_i)$, we use the **Law of Total Probability**. We integrate (or in this discrete case, sum) the joint density $p(x_i, C_i = k)$ over all possible states of the latent cluster variable $C_i$:

$$p(x_i) = \sum_{k=1}^{K} P(X_i=x_i, C_i = k)$$

Using the definition of conditional probability, we can rewrite the joint distribution as the product of the conditional distribution of $X_i$ given the cluster and the prior probability of that cluster:

$$p(x_i) = \sum_{k=1}^{K} P(X_i=x_i \mid C_i = k) P(C_i = k)$$

Given the definitions in the problem:

* The prior probability of selecting cluster $k$ is $P(C_i = k) = \phi_k$.
* The conditional distribution of the observation given it belongs to cluster $k$ is a multivariate Gaussian: $X_i \mid C_i = k \sim \mathscr{N}(\mu_k, \Sigma_k)$.

Substituting these components back into the summation yields:

$$p(x_i) = \sum_{k=1}^{K} \phi_k \mathscr{N}(x_i \mid \mu_k, \Sigma_k)$$

---

**Why is this called a Gaussian Mixture Density?**

This density is called a **Gaussian mixture density** for two main reasons:

* **Combination of Components:** It models a complex probability distribution by "mixing" together $K$ distinct, simpler multivariate Gaussian distributions (called component distributions), each with its own mean vector $\mu_k$ and covariance matrix $\Sigma_k$.
* **Weighted Representation:** The parameters $\phi_k$ act as **mixture weights** (or mixing proportions). They dictate how much influence each individual Gaussian component has on the overall distribution, satisfying the valid probability constraints where $\sum_{k=1}^K \phi_k = 1$ and $\phi_k \ge 0$.


### Part 2

Following the marginal density from Part 1, the core of framing Gaussian Mixture Clustering as a **conditional update** is calculating the posterior probability that a specific observation $x_i$ belongs to cluster $k$.

1. The Mathematical Derivation (Bayes' Theorem)

Using Bayes' Theorem, we update our prior belief $\phi_k$ after observing the data point $x_i$:

$$P(C_i = k \mid X_i=x_i) = \frac{P(X_i=x_i \mid C_i = k) P(C_i = k)}{p(x_i)}$$

Substituting the components we know:

* The prior: $P(C_i = k) = \phi_k$
* The likelihood: $p(X_i=x_i \mid C_i = k) = \mathscr{N}(x_i \mid \mu_k, \Sigma_k)$
* The marginal density (from Part 1): $p(x_i) = \sum_{j=1}^{K} \phi_j \mathscr{N}(x_i \mid \mu_j, \Sigma_j)$

This gives us the explicit posterior updating formula, often denoted as the **responsibility** $\gamma_{ik}$:

$$\gamma_{ik} = P(C_i = k \mid X_i=x_i) = \frac{\phi_k \mathscr{N}(x_i \mid \mu_k, \Sigma_k)}{\sum_{j=1}^{K} \phi_j \mathscr{N}(x_i \mid \mu_j, \Sigma_j)}$$

---

2. Interpretation as a Conditional Update

This formula beautifully illustrates the concept of **conditional updating**:

* **Prior Base:** We start with $\phi_k$, our baseline assumption of how likely any data point belongs to cluster $k$ before looking at its actual value.
* **The Evidence Likelihood:** We modulate this prior by multiplying it by $\mathscr{N}(x_i \mid \mu_k, \Sigma_k)$, which measures how well the specific location of $x_i$ matches the distribution of cluster $k$.
* **Relative Soft Assignment:** The denominator normalizes these values across all $K$ clusters. Instead of a hard, deterministic assignment, $\gamma_{ik}$ provides a "soft" fraction between 0 and 1, dynamically shifting weight toward the cluster that best explains the newly observed data point.

### Part 3

1. Showing $\mathbb{E}[Z_{ik} \mid X_i = x_i] = P(C_i = k \mid X_i = x_i)$

We have seen that, the conditional expectation of a discrete random variable given an event is the sum of its possible values weighted by their conditional probabilities.

Since $Z_{ik}$ is a binary indicator variable (taking only the values $1$ or $0$), we can write its conditional expectation explicitly:

$$\mathbb{E}[Z_{ik} \mid X_i = x_i] = \sum_{z \in \{0,1\}} z \cdot P(Z_{ik} = z \mid X_i = x_i)$$

Expanding this sum:

$$\mathbb{E}[Z_{ik} \mid X_i = x_i] = \left( 1 \cdot P(Z_{ik} = 1 \mid X_i = x_i) \right) + \left( 0 \cdot P(Z_{ik} = 0 \mid X_i = x_i) \right)$$

$$\mathbb{E}[Z_{ik} \mid X_i = x_i] = P(Z_{ik} = 1 \mid X_i = x_i)$$

Because the event $Z_{ik} = 1$ is logically identical to the event that data point $i$ belongs to cluster $k$ ($C_i = k$), we can substitute the events directly:

$$\mathbb{E}[Z_{ik} \mid X_i = x_i] = P(C_i = k \mid X_i = x_i)$$

---

2. Showing the Vector Form Equals the Vector of Responsibilities

The expectation of a random vector is simply the vector of the expectations of its individual components. Applying the result from the previous step element-by-element to the vector $Z_i$:

$$\mathbb{E}[Z_i \mid X_i = x_i] = \begin{bmatrix}
\mathbb{E}[Z_{i1} \mid X_i = x_i] \\
\mathbb{E}[Z_{i2} \mid X_i = x_i] \\
\vdots \\
\mathbb{E}[Z_{iK} \mid X_i = x_i]
\end{bmatrix} = \begin{bmatrix}
P(C_i = 1 \mid X_i = x_i) \\
P(C_i = 2 \mid X_i = x_i) \\
\vdots \\
P(C_i = K \mid X_i = x_i)
\end{bmatrix}$$

By definition, the posterior probability of cluster membership $P(C_i = k \mid X_i = x_i)$ is denoted as the responsibility $\gamma_{ik}$. Therefore, substituting $\gamma_{ik}$ gives:

$$\mathbb{E}[Z_i \mid X_i = x_i] = \begin{bmatrix}
\gamma_{i1} \\
\gamma_{i2} \\
\vdots \\
\gamma_{iK}
\end{bmatrix}$$

---

3. Conclusion

We conclude that the **soft cluster assignment** vector in a Gaussian Mixture Model (the set of responsibilities across all components) is **precisely the conditional expectation of the one-hot encoded latent random vector**, given the observed data:

$$\text{Soft Assignment} = \mathbb{E}[Z_i \mid X_i = x_i]$$

> **Intuition:** While the true underlying cluster identity $Z_i$ is a "hard" binary switch (a vector of 0s with a single 1), our best mathematical prediction of that identity after seeing the data point $x_i$ is its expected value, which smooths out into a continuous profile of probabilities between $0$ and $1$.

### Part 4

In the context of a Gaussian Mixture Model (GMM), the transition from the conditional expectation vector to the maximum posterior index represents moving from an **un-quantized, probabilistic model of uncertainty** to a **deterministic decision**.

Here is how they differ fundamentally:

1. Soft Clustering (The Conditional Expectation)

* **Mathematical Representation:** The soft assignment is given by the continuous vector of expectations $\mathbb{E}[Z_i \mid X_i = x_i] = [\gamma_{i1}, \gamma_{i2}, \dots, \gamma_{iK}]^T$, where each element $\gamma_{ik} \in [0, 1]$ and $\sum_{k=1}^K \gamma_{ik} = 1$.
* **Core Characteristics:**
* **Maintains Ambiguity:** It does not force a data point into a single category. If a point $x_i$ lies right on the overlapping boundary between Cluster 1 and Cluster 2, the model natively preserves this by outputting something like $\gamma_{i1} = 0.49$ and $\gamma_{i2} = 0.51$.
* **Partial Membership:** Every observation mathematically belongs to *all* clusters simultaneously, just with varying degrees of responsibility.
* **Downstream Utility:** This distribution is vital during model training (like the E-step of the Expectation-Maximization algorithm) because it allows gradients and parameter updates to smoothly account for data point positioning.



---

2. Hard Clustering (The $\operatorname{arg\,max}$ Decision)

* **Mathematical Representation:** The hard assignment collapses the continuous vector down to a single scalar index $\widehat C_i = \operatorname{arg\,max}_{1\le k\le K} \gamma_{ik}$, mapping the point to a discrete category $\widehat C_i \in \{1, 2, \dots, K\}$.
* **Core Characteristics:**
* **Eliminates Ambiguity:** It applies a "winner-take-all" threshold. Even if Cluster 2 only beats Cluster 1 by a microscopic margin ($0.51$ vs $0.49$), the point is assigned fully to Cluster 2.
* **Mutually Exclusive Membership:** The information about competing probabilities is discarded, treating the point as though it belongs 100% to the chosen component and 0% to all others.
* **Downstream Utility:** This is typically the final operational phase used for human interpretation, segmenting a customer base, or generating distinct categorical labels.



---

**Summary Contrast:**

| Feature | Soft Clustering ($\mathbb{E}[Z_i \mid X_i = x_i]$) | Hard Clustering ($\widehat C_i$) |
| --- | --- | --- |
| **Output Type** | Continuous probability vector ($\mathbb{R}^K$) | Discrete integer scalar |
| **Geometry** | Reflects a point's relative distance to all cluster centers | Assigns a point strictly to a bounded geometric region (Voronoi cell) |
| **Information** | Preserves classification uncertainty | Discards boundary ambiguity for a definitive choice |


### Part 5

1. Showing $\mathbb{E}[X_i \mid C_i = k] = \mu_k$

By the definition of the Gaussian Mixture Model, conditional on knowing that observation $i$ belongs to cluster $k$ ($C_i = k$), the random variable $X_i$ follows a multivariate normal (Gaussian) distribution:

$$X_i \mid C_i = k \sim \mathscr{N}(\mu_k, \Sigma_k)$$

The probability density function for a multivariate Gaussian distribution centered at $\mu_k$ is symmetric around $\mu_k$. By definition, the expected value (mean) of an explicitly parameterized normal distribution $\mathscr{N}(\mu_k, \Sigma_k)$ is its mean vector parameter.

Thus, writing it out as an expectation integral:

$$\mathbb{E}[X_i \mid C_i = k] = \int_{\mathbb{R}^d} x_i \cdot \mathscr{N}(x_i \mid \mu_k, \Sigma_k) \, dx_i = \mu_k$$

---

2. Why $\mu_k$ is interpreted as the "Center" of Cluster $k$

The parameter $\mu_k$ represents the **center of mass** or **centroid** of the $k$-th cluster distribution for several geometric and statistical reasons:

* **Point of Highest Density:** For a Gaussian distribution, the mode (the peak of the density curve) perfectly coincides with the mean. Therefore, $\mu_k$ is the single location in space where a data point from cluster $k$ is most likely to appear.
* **Symmetry & Balance:** Because the Gaussian density is spherically or elliptically symmetric around $\mu_k$, any deviations away from $\mu_k$ in opposite directions balance out perfectly, making it the geometric balancing point of the cluster.

---
3. Comparing the Two Conditional Expectations

The two expressions flip the conditioning variable and the target variable, leading to completely opposite interpretations:

Expression A: $\mathbb{E}[Z_i \mid X_i = x_i]$ (Soft Cluster Membership)

* **What is fixed?** The data point's location ($X_i = x_i$) is known and fixed.
* **What is random?** The cluster identity ($Z_i$) is unknown.
* **Why it yields membership:** As proven in Part 3, this expectation outputs a vector of probabilities (responsibilities $\gamma_{ik}$). It answers the question: *"Given that an observed point landed exactly at position $x_i$, what fraction of our belief belongs to each cluster?"* It maps a coordinate space to a probability space.

Expression B: $\mathbb{E}[X_i \mid C_i = k]$ (Mean Location of a Cluster)

* **What is fixed?** The cluster identity ($C_i = k$) is known and fixed.
* **What is random?** The coordinate location ($X_i$) of the data point is unknown.
* **Why it yields the mean location:** This expectation outputs a spatial coordinate vector ($\mu_k$). It answers the question: *"If we generate a hypothetical data point purely out of cluster $k$, what is its expected position in space?"* It maps a discrete cluster identity to a physical coordinate space.



### Part 6

1. Showing the Logarithm of the Complete-Data Likelihood

We begin with the complete-data likelihood function, assuming the latent indicators $z_{ik}$ are known:

$$p(x_1,\dots,x_n,z_1,\dots,z_n)=\prod_{i=1}^n \prod_{k=1}^K \left[ \phi_k \mathscr{N}(x_i\mid \mu_k,\Sigma_k) \right]^{z_{ik}}$$

Taking the natural logarithm ($\log$) of both sides allows us to convert the products into sums:

$$\ell_c = \log \left( \prod_{i=1}^n \prod_{k=1}^K \left[ \phi_k \mathscr{N}(x_i\mid \mu_k,\Sigma_k) \right]^{z_{ik}} \right)$$

Using the logarithmic property $\log(\prod A) = \sum \log(A)$, we move the products outside as summation signs:

$$\ell_c = \sum_{i=1}^n \sum_{k=1}^K \log \left( \left[ \phi_k \mathscr{N}(x_i\mid \mu_k,\Sigma_k) \right]^{z_{ik}} \right)$$

Using the power property of logarithms $\log(A^b) = b \log(A)$, we bring the exponent $z_{ik}$ to the front of each log term:

$$\ell_c = \sum_{i=1}^n \sum_{k=1}^K z_{ik} \log \left( \phi_k \mathscr{N}(x_i\mid \mu_k,\Sigma_k) \right)$$

Finally, applying the product rule $\log(AB) = \log A + \log B$ inside the bracket yields the desired expression:

$$\ell_c=\sum_{i=1}^n \sum_{k=1}^K z_{ik} \left[ \log \phi_k + \log \mathscr{N}(x_i\mid \mu_k,\Sigma_k) \right]$$

---

2. Why this Expression is Easy to Maximize if $z_{ik}$ Were Known

If the latent indicators $z_{ik}$ were observed constants rather than hidden variables, optimizing this log-likelihood would collapse from a complex, coupled mixture problem into a series of straightforward independent problems:

* **Decoupling of Components:** Notice that the log-likelihood is a linear combination where each individual cluster component $k$ is completely separated from the others. There are no sums inside the logarithms (unlike the standard marginal log-likelihood).
* **Independent Parameters:**
* To find the optimal mean $\mu_k$ and covariance $\Sigma_k$ for a specific cluster $k$, we only need to maximize the terms where $z_{ik} = 1$. This means we can isolate the data points assigned to cluster $k$ and compute their standard empirical sample mean and sample variance independently of any other clusters.
* The mixing weights $\phi_k$ can be optimized independently using a simple Lagrange multiplier to ensure they sum to 1, leading directly to the fraction of points in each cluster.



In short, knowing $z_{ik}$ turns a hard mixture problem into $K$ separate, standard maximum likelihood estimation (MLE) tasks for individual Gaussian distributions.

### Part 7

1. The Expected Complete-Data Log-Likelihood ($Q$-function)

In the Expectation-Maximization (EM) algorithm, because we cannot compute the complete-data log-likelihood $\ell_c$ directly (since the true binary indicators $z_{ik}$ are hidden), we instead compute its **conditional expectation** with respect to the posterior distribution of the latent variables, given the observed data $x_i$ and the current estimates of our parameters.

From Part 6, the complete-data log-likelihood is:


$$\ell_c = \sum_{i=1}^n \sum_{k=1}^K z_{ik} \left[ \log \phi_k + \log \mathscr{N}(x_i \mid \mu_k, \Sigma_k) \right]$$

Taking the conditional expectation of both sides given the observed data $X = x$:


$$Q = \mathbb{E}[\ell_c \mid X = x] = \sum_{i=1}^n \sum_{k=1}^K \mathbb{E}[Z_{ik} \mid X_i = x_i] \left[ \log \phi_k + \log \mathscr{N}(x_i \mid \mu_k, \Sigma_k) \right]$$

As proved in Part 3, the conditional expectation of the binary indicator is precisely the posterior cluster responsibility: $\mathbb{E}[Z_{ik} \mid X_i = x_i] = \gamma_{ik}$. Substituting this yields the $Q$-function formulation:

$$Q = \sum_{i=1}^n \sum_{k=1}^K \gamma_{ik} \left[ \log \phi_k + \log \mathscr{N}(x_i \mid \mu_k, \Sigma_k) \right]$$

---

2. Why the E-Step is a "Conditional Update" of Cluster Membership Probabilities

The Expectation step (E-step) is fundamentally a **conditional update** rather than a static computation for the following reasons:

* **Information Incorporation:** We begin with a baseline structural definition of our clusters (parameterized by the current $\phi_k, \mu_k, \Sigma_k$). When an empirical data point $x_i$ is observed, the E-step dynamically alters our prior beliefs ($\phi_k$) about that point's identity.
* **Bayesian Revision:** It uses Bayes' Theorem to find the probability distribution of the latent variables conditioned *specifically* on the realized data. The responsibilities ($\gamma_{ik}$) are not fixed constants; they are updated outputs that shift dynamically depending on how well the current cluster configurations explain the data coordinates $x_i$.
* **Passing the Torch:** This step converts our absolute ignorance about the true cluster labels into an optimal, smoothed probabilistic assignment vector. This conditional profile directly dictates how much weight each point will carry when rebuilding the cluster definitions in the upcoming maximization (M) step.

### Part 8

Once we have computed the posterior probabilities (responsibilities $\gamma_{ik}$) in the conditional update step (the E-step), the final phase of framing Gaussian Mixture Clustering as an iterative update process is **updating the model parameters** ($\phi_k, \mu_k, \Sigma_k$) to maximize the expected log-likelihood (the M-step).

Here is how the parameters are updated conditionally based on the responsibilities:

1. Effective Number of Points in a Cluster ($N_k$)

First, we compute the total responsibility assigned to cluster $k$ across all $N$ data points. This represents the effective number of observations belonging to cluster $k$:

$$N_k = \sum_{i=1}^{N} \gamma_{ik}$$

---

2. Updating the Mixture Weights ($\phi_k$)

The prior probability or mixing proportion for cluster $k$ is updated as the fraction of the total dataset assigned to that cluster:

$$\phi_k^{\text{new}} = \frac{N_k}{N}$$

---

3. Updating the Cluster Means ($\mu_k$)

The mean vector of each Gaussian component is updated by taking a weighted average of all data points, where the weights are the responsibilities:

$$\mu_k^{\text{new}} = \frac{1}{N_k} \sum_{i=1}^{N} \gamma_{ik} x_i$$

---

4. Updating the Covariance Matrices ($\Sigma_k$)

Similarly, the covariance matrix for each component is updated by computing the sample covariance of the data points around the *new* mean $\mu_k^{\text{new}}$, weighted by the responsibilities:

$$\Sigma_k^{\text{new}} = \frac{1}{N_k} \sum_{i=1}^{N} \gamma_{ik} (x_i - \mu_k^{\text{new}})(x_i - \mu_k^{\text{new}})^T$$

---

**Interpretation as a Global System Update:**

In this framework, the clustering process operates as a two-stage conditional updating engine:

1. **Local Update (Part 2):** Fix the cluster parameters and update our beliefs about which data points belong to which clusters ($\gamma_{ik}$).
2. **Global Update (Part 3):** Fix those point assignments ($\gamma_{ik}$) and update the structural parameters of the clusters to better fit the data.

Let me know if there's a **Part 4** or an implementation step you'd like to look at next!

### Part 9

Gaussian Mixture Model (GMM) clustering can be viewed as a repeated process of conditional updating because it iteratively refines its structural assumptions as new data evidence is ingested. At the start of an iteration, the mixture weight $\phi_k$ acts as our baseline **prior probability** of selecting cluster $k$. Upon encountering an observed data point $x_i$, the model evaluates the Gaussian density $\mathscr{N}(x_i \mid \mu_k, \Sigma_k)$, which mathematically **measures how compatible** $x_i$ is with that specific cluster's current shape and location. Combining this prior and likelihood via Bayes' Theorem yields the responsibility $\gamma_{ik}$, representing the **posterior probability** that the data point belongs to cluster $k$ given its observed position.

Aggregated across all components, this collection of responsibilities defines the **soft assignment vector** $\mathbb{E}[Z_i \mid X_i = x_i]$, capturing our complete, updated state of uncertainty. Finally, the M-step closes the loop by **updating the cluster parameters** ($\phi_k, \mu_k, \Sigma_k$) using these newly updated posterior membership probabilities as weights to re-center and reshape the components. From this cyclical process, we can conclude that Gaussian mixture clustering is fundamentally a form of **probabilistic clustering based on conditional expectations of latent cluster membership variables**.

### Part 10

In [10]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = ""

# Load the latest version
# df = kagglehub.load_dataset(
#   KaggleDatasetAdapter.PANDAS,
#   "arjunbhasin2013/ccdata",
#   file_path,
#   # Provide any additional arguments like
#   # sql_query or pandas_kwargs. See the
#   # documenation for more information:
#   # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
# )
df = pd.read_csv("/content/CC GENERAL.csv")

print("First 5 records:", df.head())

First 5 records:   CUST_ID      BALANCE  BALANCE_FREQUENCY  PURCHASES  ONEOFF_PURCHASES  \
0  C10001    40.900749           0.818182      95.40              0.00   
1  C10002  3202.467416           0.909091       0.00              0.00   
2  C10003  2495.148862           1.000000     773.17            773.17   
3  C10004  1666.670542           0.636364    1499.00           1499.00   
4  C10005   817.714335           1.000000      16.00             16.00   

   INSTALLMENTS_PURCHASES  CASH_ADVANCE  PURCHASES_FREQUENCY  \
0                    95.4      0.000000             0.166667   
1                     0.0   6442.945483             0.000000   
2                     0.0      0.000000             1.000000   
3                     0.0    205.788017             0.083333   
4                     0.0      0.000000             0.083333   

   ONEOFF_PURCHASES_FREQUENCY  PURCHASES_INSTALLMENTS_FREQUENCY  \
0                    0.000000                          0.083333   
1                  

In [ ]:
import kagglehub

# Download latest version
kagglepath="arjunbhasin2013/ccdata"
path = kagglehub.dataset_download(kagglepath)

print("Path to dataset files:", path)

In [13]:
import os
import pandas as pd
import kagglehub

path = kagglehub.dataset_download("arjunbhasin2013/ccdata")

print(f"Dataset directory: {path}")
print("Files:", os.listdir(path))

file_path = os.path.join(path, "CC GENERAL.csv")
df2 = pd.read_csv(file_path)

print("Dataset shape:", df2.shape)
display(df2.head())

100%|██████████| 340k/340k [00:00<00:00, 72.7MB/s]

Extracting files...
Dataset directory: /root/.cache/kagglehub/datasets/arjunbhasin2013/ccdata/versions/1
Files: ['CC GENERAL.csv']
Dataset shape: (8950, 18)


,CUST_ID,BALANCE,BALANCE_FREQUENCY,PURCHASES,ONEOFF_PURCHASES,INSTALLMENTS_PURCHASES,CASH_ADVANCE,PURCHASES_FREQUENCY,ONEOFF_PURCHASES_FREQUENCY,PURCHASES_INSTALLMENTS_FREQUENCY,CASH_ADVANCE_FREQUENCY,CASH_ADVANCE_TRX,PURCHASES_TRX,CREDIT_LIMIT,PAYMENTS,MINIMUM_PAYMENTS,PRC_FULL_PAYMENT,TENURE
0,C10001,40.900749,0.818182,95.40,0.00,95.4,0.000000,0.166667,0.000000,0.083333,0.000000,0,2,1000.0,201.802084,139.509787,0.000000,12
1,C10002,3202.467416,0.909091,0.00,0.00,0.0,6442.945483,0.000000,0.000000,0.000000,0.250000,4,0,7000.0,4103.032597,1072.340217,0.222222,12
2,C10003,2495.148862,1.000000,773.17,773.17,0.0,0.000000,1.000000,1.000000,0.000000,0.000000,0,12,7500.0,622.066742,627.284787,0.000000,12
3,C10004,1666.670542,0.636364,1499.00,1499.00,0.0,205.788017,0.083333,0.083333,0.000000,0.083333,1,1,7500.0,0.000000,NaN,0.000000,12
4,C10005,817.714335,1.000000,16.00,16.00,0.0,0.000000,0.083333,0.083333,0.000000,0.000000,0,1,1200.0,678.334763,244.791237,0.000000,12


In [ ]:
df2

In [14]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


class GMMFinancialSegmenter:

    def __init__(self, n_components=3, random_state=42):
        """Initializes the GMM Segmenter framework."""
        self.n_components = n_components
        self.random_state = random_state
        self.scaler = StandardScaler()
        self.model = GaussianMixture(
            n_components=self.n_components,
            covariance_type="full",
            random_state=self.random_state,
        )

    def prepare_data(self, df, feature_cols, test_size=0.2):
        """Extracts features, normalizes them, and splits into train/test sets."""
        X = df[feature_cols].dropna().values

        # Scale features (Crucial for GMM since variance shapes depend on scale)
        X_scaled = self.scaler.fit_transform(X)

        # Train/Test split
        X_train, X_test = train_test_split(
            X_scaled, test_size=test_size, random_state=self.random_state
        )
        return X_train, X_test

    def fit(self, X_train):
        """Trains the Gaussian Mixture Model on training data."""
        self.model.fit(X_train)
        print(" GMM Training complete.")
        print(f"Converged: {self.model.converged_}")
        print(f"Iterations taken: {self.model.n_iter_}")

    def evaluate(self, X_test):
        """Validates the model on test data using average log-likelihood."""
        # score() returns the per-sample average log-likelihood
        avg_log_likelihood = self.model.score(X_test)
        print(f"\n Validation Performance (Test Set):")
        print(f"Average Log-Likelihood: {avg_log_likelihood:.4f}")
        return avg_log_likelihood

    def plot_density_heatmap(self, X_train, feature_names):
        """Generates a 2D density heatmap of the empirical training distribution."""
        # Invert scaling back to original currency values for realistic viewing
        X_orig = self.scaler.inverse_transform(X_train)

        fig = px.density_heatmap(
            x=X_orig[:, 0],
            y=X_orig[:, 1],
            labels={"x": feature_names[0], "y": feature_names[1]},
            title="Empirical Training Data Density Heatmap",
            marginal_x="histogram",
            marginal_y="histogram"
        )
        # Explicitly update the colorscale for the main heatmap trace
        fig.update_traces(colorscale="Viridis", selector=dict(type='histogram2d'))
        fig.update_layout(template="plotly_white")
        fig.show()

    def _generate_contour_base(self, X_data):
        """Helper to compute coordinate grids and posterior probability maps."""
        x_min, x_max = X_data[:, 0].min() - 0.5, X_data[:, 0].max() + 0.5
        y_min, y_max = X_data[:, 1].min() - 0.5, X_data[:, 1].max() + 0.5
        xx, yy = np.meshgrid(
            np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200)
        )

        grid_points = np.c_[xx.ravel(), yy.ravel()]
        responsibilities = self.model.predict_proba(grid_points)
        max_prob = responsibilities.max(axis=1).reshape(xx.shape)

        grid_orig = self.scaler.inverse_transform(grid_points)
        xx_orig = grid_orig[:, 0].reshape(xx.shape)
        yy_orig = grid_orig[:, 1].reshape(yy.shape)

        return xx_orig, yy_orig, max_prob

    def plot_training_assignments(self, X_train, feature_names):
        """Plots the training data points over the soft assignment confidence boundaries."""
        xx_orig, yy_orig, max_prob = self._generate_contour_base(X_train)
        hard_labels = self.model.predict(X_train)
        X_train_orig = self.scaler.inverse_transform(X_train)

        fig = go.Figure()
        fig.add_trace(
            go.Contour(
                x=xx_orig[0, :], y=yy_orig[:, 0], z=max_prob,
                colorscale="Cividis", contours_coloring="heatmap",
                name="Confidence Level", hoverinfo="skip", opacity=0.6,
            )
        )

        for k in range(self.n_components):
            cluster_mask = hard_labels == k
            fig.add_trace(
                go.Scatter(
                    x=X_train_orig[cluster_mask, 0], y=X_train_orig[cluster_mask, 1],
                    mode="markers", name=f"Train Cluster {k+1}",
                    marker=dict(size=6, line=dict(width=1, color="Black")),
                )
            )

        fig.update_layout(
            title="GMM Soft-Assignment Confidence Boundaries on Training Data",
            xaxis_title=feature_names[0], yaxis_title=feature_names[1],
            template="plotly_white",
        )
        fig.show()

    def plot_soft_assignments(self, X_test, feature_names):
        """Plots the test data points overlaying the continuous soft assignment profiles."""
        xx_orig, yy_orig, max_prob = self._generate_contour_base(X_test)
        hard_labels = self.model.predict(X_test)
        X_test_orig = self.scaler.inverse_transform(X_test)

        fig = go.Figure()
        fig.add_trace(
            go.Contour(
                x=xx_orig[0, :], y=yy_orig[:, 0], z=max_prob,
                colorscale="Cividis", contours_coloring="heatmap",
                name="Confidence Level", hoverinfo="skip", opacity=0.6,
            )
        )

        for k in range(self.n_components):
            cluster_mask = hard_labels == k
            fig.add_trace(
                go.Scatter(
                    x=X_test_orig[cluster_mask, 0], y=X_test_orig[cluster_mask, 1],
                    mode="markers", name=f"Test Cluster {k+1}",
                    marker=dict(size=6, line=dict(width=1, color="Black")),
                )
            )

        fig.update_layout(
            title="GMM Soft-Assignment Confidence Boundaries on Test Data",
            xaxis_title=feature_names[0], yaxis_title=feature_names[1],
            template="plotly_white",
        )
        fig.show()


# =====================================================================
# Execution Block: Using Kaggle Credit Card Data Columns
# =====================================================================
if __name__ == "__main__":
    np.random.seed(42)
    synthetic_data = {
        "PURCHASES": np.hstack([
            np.random.exponential(400, 400),
            np.random.normal(2500, 600, 300),
            np.random.normal(6000, 1200, 100),
        ]),
        "CREDIT_LIMIT": np.hstack([
            np.random.normal(2000, 800, 400),
            np.random.normal(7000, 1500, 300),
            np.random.normal(12000, 2000, 100),
        ]),
    }
    df = df2 #pd.DataFrame(synthetic_data)
    features = ["PURCHASES", "CREDIT_LIMIT"]

    segmenter = GMMFinancialSegmenter(n_components=3)
    X_train, X_test = segmenter.prepare_data(df, features)
    segmenter.fit(X_train)
    segmenter.evaluate(X_test)

    # Interactive Plots
    segmenter.plot_density_heatmap(X_train, features)
    segmenter.plot_training_assignments(X_train, features)  # New training assignment plot
    segmenter.plot_soft_assignments(X_test, features)

 GMM Training complete.
Converged: True
Iterations taken: 19

 Validation Performance (Test Set):
Average Log-Likelihood: -1.6465


In [8]:
import os
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from sklearn.impute import SimpleImputer
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


class GMMFinancialSegmenter:
    """Two-dimensional GMM segmentation with train/test validation."""

    def __init__(
        self,
        n_components=3,
        test_size=0.20,
        random_state=42,
        covariance_type="full",
        grid_size=180,
    ):
        self.n_components = n_components
        self.test_size = test_size
        self.random_state = random_state
        self.covariance_type = covariance_type
        self.grid_size = grid_size

        self.imputer = SimpleImputer(strategy="median")
        self.scaler = StandardScaler()
        self.model = GaussianMixture(
            n_components=n_components,
            covariance_type=covariance_type,
            n_init=10,
            max_iter=500,
            reg_covar=1e-6,
            random_state=random_state,
        )

    def fit(self, data, features=("PURCHASES", "CREDIT_LIMIT")):
        self.features = list(features)

        missing_columns = [c for c in self.features if c not in data.columns]
        if missing_columns:
            raise ValueError(f"Missing required columns: {missing_columns}")

        selected = data[self.features].apply(pd.to_numeric, errors="coerce")
        selected = selected.replace([np.inf, -np.inf], np.nan)

        self.X_train_raw, self.X_test_raw = train_test_split(
            selected,
            test_size=self.test_size,
            random_state=self.random_state,
        )

        train_imputed = self.imputer.fit_transform(self.X_train_raw)
        test_imputed = self.imputer.transform(self.X_test_raw)

        self.X_train_clean = pd.DataFrame(
            train_imputed,
            columns=self.features,
            index=self.X_train_raw.index,
        )
        self.X_test_clean = pd.DataFrame(
            test_imputed,
            columns=self.features,
            index=self.X_test_raw.index,
        )

        self.X_train_scaled = self.scaler.fit_transform(self.X_train_clean)
        self.X_test_scaled = self.scaler.transform(self.X_test_clean)

        self.model.fit(self.X_train_scaled)

        self.train_responsibilities = self.model.predict_proba(
            self.X_train_scaled
        )
        self.test_responsibilities = self.model.predict_proba(
            self.X_test_scaled
        )
        self.train_labels = np.argmax(self.train_responsibilities, axis=1)
        self.test_labels = np.argmax(self.test_responsibilities, axis=1)
        self.train_confidence = np.max(self.train_responsibilities, axis=1)
        self.test_confidence = np.max(self.test_responsibilities, axis=1)

        self.train_average_log_likelihood = self.model.score(
            self.X_train_scaled
        )
        self.test_average_log_likelihood = self.model.score(
            self.X_test_scaled
        )

        print(f"Converged: {self.model.converged_}")
        print(f"EM iterations: {self.model.n_iter_}")
        print(
            "Training average log-likelihood: "
            f"{self.train_average_log_likelihood:.4f}"
        )
        print(
            "Test average log-likelihood: "
            f"{self.test_average_log_likelihood:.4f}"
        )
        print(f"Training observations: {len(self.X_train_clean)}")
        print(f"Test observations: {len(self.X_test_clean)}")

        return self

    def _responsibility_grid(self):
        x_name, y_name = self.features
        combined = pd.concat([self.X_train_clean, self.X_test_clean])

        x_low, x_high = np.quantile(combined[x_name], [0.005, 0.995])
        y_low, y_high = np.quantile(combined[y_name], [0.005, 0.995])

        x_padding = 0.05 * max(x_high - x_low, 1.0)
        y_padding = 0.05 * max(y_high - y_low, 1.0)
        x_values = np.linspace(x_low - x_padding, x_high + x_padding, self.grid_size)
        y_values = np.linspace(y_low - y_padding, y_high + y_padding, self.grid_size)
        xx, yy = np.meshgrid(x_values, y_values)

        raw_grid = pd.DataFrame(
            np.column_stack([xx.ravel(), yy.ravel()]),
            columns=self.features,
        )
        scaled_grid = self.scaler.transform(raw_grid)
        responsibilities = self.model.predict_proba(scaled_grid)

        confidence = responsibilities.max(axis=1).reshape(xx.shape)
        winning_cluster = responsibilities.argmax(axis=1).reshape(xx.shape)

        return x_values, y_values, confidence, winning_cluster

    def plot_density_heatmap(self):
        x_name, y_name = self.features
        plot_data = self.X_train_clean.copy()

        fig = px.density_heatmap(
            plot_data,
            x=x_name,
            y=y_name,
            nbinsx=60,
            nbinsy=60,
            marginal_x="histogram",
            marginal_y="histogram",
            color_continuous_scale=px.colors.sequential.Viridis,
            title="Empirical 2D Density of the Training Data",
        )
        fig.update_layout(template="plotly_white", height=700)
        return fig

    def _assignment_figure(self, split="train"):
        x_values, y_values, confidence, winning_cluster = (
            self._responsibility_grid()
        )

        if split == "train":
            observations = self.X_train_clean
            labels = self.train_labels
            point_confidence = self.train_confidence
            title = "Training Assignments and Posterior-Confidence Surface"
        elif split == "test":
            observations = self.X_test_clean
            labels = self.test_labels
            point_confidence = self.test_confidence
            title = "Out-of-Sample Test Assignments and Posterior-Confidence Surface"
        else:
            raise ValueError("split must be 'train' or 'test'")

        fig = go.Figure()

        # Continuous maximum responsibility: max_k gamma_k(x_grid)
        fig.add_trace(go.Contour(
            x=x_values,
            y=y_values,
            z=confidence,
            colorscale="Blues",
            opacity=0.68,
            contours=dict(start=0.34, end=1.0, size=0.06, coloring="heatmap"),
            colorbar=dict(title="max γₖ"),
            hovertemplate=(
                f"{self.features[0]}=%{{x:.2f}}<br>"
                f"{self.features[1]}=%{{y:.2f}}<br>"
                "Maximum responsibility=%{z:.3f}<extra></extra>"
            ),
            name="Posterior confidence",
        ))

        # Cluster boundaries occur where the winning component changes.
        fig.add_trace(go.Contour(
            x=x_values,
            y=y_values,
            z=winning_cluster,
            contours=dict(
                start=-0.5,
                end=self.n_components - 0.5,
                size=1,
                coloring="none",
                showlabels=False,
            ),
            line=dict(color="black", width=1.5),
            showscale=False,
            hoverinfo="skip",
            name="Decision boundaries",
        ))

        hover_text = [
            f"Cluster={label}<br>max γ={conf:.3f}"
            for label, conf in zip(labels, point_confidence)
        ]

        fig.add_trace(go.Scattergl(
            x=observations[self.features[0]],
            y=observations[self.features[1]],
            mode="markers",
            marker=dict(
                color=labels,
                colorscale="Turbo",
                cmin=0,
                cmax=self.n_components - 1,
                size=5,
                opacity=0.55,
                line=dict(width=0),
            ),
            text=hover_text,
            hovertemplate=(
                f"{self.features[0]}=%{{x:.2f}}<br>"
                f"{self.features[1]}=%{{y:.2f}}<br>"
                "%{text}<extra></extra>"
            ),
            name=f"{split.title()} observations",
        ))

        fig.update_layout(
            title=title,
            xaxis_title=self.features[0],
            yaxis_title=self.features[1],
            template="plotly_white",
            height=650,
            hovermode="closest",
        )
        return fig

    def plot_training_assignments(self):
        return self._assignment_figure(split="train")

    def plot_test_assignments(self):
        return self._assignment_figure(split="test")

    def cluster_summary(self):
        summary_data = self.X_train_clean.copy()
        summary_data["Cluster"] = self.train_labels
        summary_data["Maximum responsibility"] = self.train_confidence

        summary = summary_data.groupby("Cluster").agg(
            Observations=("Cluster", "size"),
            Mean_PURCHASES=(self.features[0], "mean"),
            Median_PURCHASES=(self.features[0], "median"),
            Mean_CREDIT_LIMIT=(self.features[1], "mean"),
            Median_CREDIT_LIMIT=(self.features[1], "median"),
            Mean_Max_Responsibility=("Maximum responsibility", "mean"),
        )
        return summary


In [9]:
# Locate the attached dataset in Colab or in the downloaded workspace.
candidate_paths = [
    "CC GENERAL.csv",
    "/content/CC GENERAL.csv",
    "upload/CC GENERAL.csv",
]

dataset_path = next((p for p in candidate_paths if os.path.exists(p)), None)

if dataset_path is None:
    try:
        from google.colab import files
        print("Upload CC GENERAL.csv")
        uploaded = files.upload()
        dataset_path = next(iter(uploaded))
    except ImportError as exc:
        raise FileNotFoundError(
            "Place 'CC GENERAL.csv' beside the notebook and run this cell again."
        ) from exc

credit_data = pd.read_csv(dataset_path)

print("Dataset shape:", credit_data.shape)
print("Missing values in selected features:")
print(credit_data[["PURCHASES", "CREDIT_LIMIT"]].isna().sum())

segmenter = GMMFinancialSegmenter(
    n_components=3,
    test_size=0.20,
    random_state=42,
    grid_size=180,
)

segmenter.fit(
    credit_data,
    features=("PURCHASES", "CREDIT_LIMIT"),
)

display(segmenter.cluster_summary().round(3))

fig_density = segmenter.plot_density_heatmap()
fig_density.show()

fig_train = segmenter.plot_training_assignments()
fig_train.show()

fig_test = segmenter.plot_test_assignments()
fig_test.show()


Dataset shape: (8950, 18)
Missing values in selected features:
PURCHASES       0
CREDIT_LIMIT    1
dtype: int64
Converged: True
EM iterations: 19
Training average log-likelihood: -1.5916
Test average log-likelihood: -1.5941
Training observations: 7160
Test observations: 1790


,Observations,Mean_PURCHASES,Median_PURCHASES,Mean_CREDIT_LIMIT,Median_CREDIT_LIMIT,Mean_Max_Responsibility
Cluster,,,,,,
0,3283,187.376,117.24,2062.915,1800.0,0.925
1,3251,948.097,830.00,5974.231,6000.0,0.915
2,626,5444.371,4271.57,9970.208,9000.0,0.920


### Interpretation of the Computational Results

The heatmap reveals that the two financial features are strongly right-skewed: most customers have relatively modest purchases and credit limits, while a smaller number occupy high-value regions. Multiple concentrations in the joint feature space motivate a mixture model rather than a single Gaussian density.

In the assignment plots, the color of each observation represents its hard label

$$
\widehat C_i=\operatorname{argmax}_{k}\gamma_{ik}.
$$

The continuous background instead displays

$$
\max_k\gamma_k(x_{\mathrm{grid}}),
$$

which is the largest component of the soft-assignment expectation vector

$$
\mathbb{E}[Z\mid X=x_{\mathrm{grid}}]
=
\begin{bmatrix}
\gamma_1(x_{\mathrm{grid}})\\
\gamma_2(x_{\mathrm{grid}})\\
\gamma_3(x_{\mathrm{grid}})
\end{bmatrix}.
$$

High background values indicate that one cluster clearly dominates. Lower values occur in overlap regions where two or more components assign comparable probabilities. Thus, the background visualizes the confidence associated with the conditional expectation vector, while the boundary lines show where its largest component changes identity.

The training plot shows the regions learned by EM. The test plot applies those same fitted parameters to observations that were not used during estimation. Similar patterns across the two plots indicate that the learned mixture structure generalizes reasonably to unseen observations. The printed test average log-likelihood provides a numerical out-of-sample density measure.

Cluster numbers are arbitrary labels rather than ordered customer-quality scores. Their practical meanings should be assigned only after reviewing the cluster summary. For example, a component with comparatively high purchases and high credit limits may be described as a high-value segment, whereas a component with low purchases may represent low-engagement customers. These descriptions are empirical and should not be treated as causal conclusions.

Finally, a GMM assumes approximately Gaussian component shapes after preprocessing. The extreme skewness and outliers in financial variables may make transformations such as $\log(1+x)$ useful in a further model comparison. The requested implementation uses standardization directly so that it follows the assignment specification.
